# Day 15 — Historical Baseline Model

Train and freeze the historical Dataset B baseline using the pre-specified protocol.

In [ ]:
# Imports
import json
from pathlib import Path

In [ ]:
# Project paths
def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "README.md").exists():
            return path
    raise RuntimeError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

def resolve_artifact_paths(value):
    """Rebase legacy project paths in loaded metadata; never rewrite artifacts.

    Keep all non-path values, model identifiers, and scientific settings intact.
    Support the original project name and the public repository name.
    """
    if isinstance(value, dict):
        return {key: resolve_artifact_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_artifact_paths(item) for item in value]
    if isinstance(value, str):
        normalized = value.replace("\\", "/")
        for name in ("FinGPT_Project", "FinGPT-Temporal-Generalization"):
            marker = "/" + name + "/"
            if marker in normalized and "\n" not in normalized:
                # Only standalone filesystem references, never prose containing a path.
                prefix, relative = normalized.split(marker, 1)
                if prefix.startswith("/") or (len(prefix) > 1 and prefix[1] == ":"):
                    if not any(char.isspace() for char in relative):
                        return str(PROJECT_ROOT / relative)
    return value

def load_artifact_json(stream, *args, **kwargs):
    return resolve_artifact_paths(json.load(stream, *args, **kwargs))


In [ ]:
# Reproducibility settings
# Original seeds and deterministic settings remain at their original workflow
# positions below, including stage-specific RNG resets. No new seed is introduced.

## Objective

Day 15 trains a new LoRA model, \(M_H\), using **only the frozen historical portion of Dataset B** defined on Day 14.

The purpose of this stage is **not** to evaluate temporal degradation yet. Instead, Day 15 creates a reproducible historical model that will later be evaluated on the strictly future F1–F6 windows.

The frozen data structure is:

- **Historical Train:** 13,539 observations
- **Historical Validation:** 5,859 observations
- **Future:** 20,002 observations
- **Validation start:** 2025-11-01 UTC
- **Future cutoff:** 2026-01-01 UTC
- **Training budget:** exactly 200 steps
- **Checkpoint selection criterion:** minimum historical validation loss

## Scientific Design

Two LoRA models are now kept conceptually separate:

- \(M_A\): Dataset-A-trained LoRA from Day 04  
  - Role: external OOD / cross-dataset robustness baseline

- \(M_H\): Historical-Dataset-B-trained LoRA created on Day 15  
  - Role: primary within-Dataset-B temporal generalization model

The intended workflow is:

\[
\text{Historical Train}
\rightarrow
\text{LoRA Training}
\rightarrow
\text{Historical Validation}
\rightarrow
\text{Checkpoint Selection}
\rightarrow
\text{Model Freeze}
\rightarrow
\text{Future F1--F6 Evaluation on Day 16}
\]

## Strict Future-Data Firewall

During Day 15, the 20,002 future observations remain fully sealed.

Future data must not influence:

- model training,
- checkpoint selection,
- hyperparameters,
- prompt design,
- class weighting,
- training duration,
- threshold selection,
- or any other modeling decision.

No F1–F6 performance metric will be computed during Day 15.

## Day 15 Terminal Goal

If all frozen protocol checks, training, validation, checkpoint selection, artifact saving, and model-freeze checks succeed, the intended final status is:

**PASS — HISTORICAL LoRA TRAINED AND FROZEN**

In [ ]:
# Initialize paths and verify Day 14 frozen artifacts

from pathlib import Path

# Project directories

# PROJECT_ROOT is discovered in the project-path setup cell.

DAY14_RESULT_DIR = PROJECT_ROOT / "results" / "day14"
DAY15_RESULT_DIR = PROJECT_ROOT / "results" / "day15"
DAY15_MODEL_DIR  = PROJECT_ROOT / "models" / "day15_historical_datasetB_lora"
NOTEBOOK_DIR     = PROJECT_ROOT / "notebooks"

# Create Day 15 output directories only.
# Do NOT modify Day 14 artifacts.
DAY15_RESULT_DIR.mkdir(parents=True, exist_ok=True)
DAY15_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Frozen Day 14 artifacts required by Day 15
# ------------------------------------------------------------
day14_required_artifacts = {
    "benchmark_population_manifest":
        DAY14_RESULT_DIR / "benchmark_population_manifest.json",

    "chronological_split_spec":
        DAY14_RESULT_DIR / "chronological_split_spec.json",

    "future_window_manifest":
        DAY14_RESULT_DIR / "future_window_manifest.csv",

    "source_composition_by_split":
        DAY14_RESULT_DIR / "source_composition_by_split.csv",

    "lora_training_protocol":
        DAY14_RESULT_DIR / "lora_training_protocol.json",

    "evaluation_protocol":
        DAY14_RESULT_DIR / "evaluation_protocol.json",

    "final_boundary_leakage_audit":
        DAY14_RESULT_DIR / "final_boundary_leakage_audit.json",
}

# ------------------------------------------------------------
# Existence check
# ------------------------------------------------------------
artifact_status = {
    name: path.exists()
    for name, path in day14_required_artifacts.items()
}

print("=" * 96)
print("DAY 15 — PATH INITIALIZATION & DAY 14 ARTIFACT CHECK")
print("=" * 96)

print(f"Project root      : {PROJECT_ROOT}")
print(f"Day 14 results    : {DAY14_RESULT_DIR}")
print(f"Day 15 results    : {DAY15_RESULT_DIR}")
print(f"Day 15 model dir  : {DAY15_MODEL_DIR}")
print()

for name, exists in artifact_status.items():
    status = "PASS" if exists else "MISSING"
    print(f"{status:<7} — {name}")

print("-" * 96)

all_required_exist = all(artifact_status.values())

print(
    "Day 14 frozen artifact checkpoint:",
    "PASS" if all_required_exist else "FAIL"
)

if not all_required_exist:
    missing = [
        str(day14_required_artifacts[name])
        for name, exists in artifact_status.items()
        if not exists
    ]

    print("\nMissing files:")
    for path in missing:
        print(" -", path)

print("=" * 96)

assert all_required_exist, (
    "Day 15 STOP: one or more required Day 14 frozen artifacts are missing."
)

In [ ]:
# Read Day 14 frozen protocol and verify core split specs

import json
import pandas as pd

# Load frozen Day 14 protocol artifacts

with open(day14_required_artifacts["benchmark_population_manifest"], "r") as f:
    benchmark_manifest = load_artifact_json(f)

with open(day14_required_artifacts["chronological_split_spec"], "r") as f:
    split_spec = load_artifact_json(f)

with open(day14_required_artifacts["lora_training_protocol"], "r") as f:
    lora_protocol = load_artifact_json(f)

with open(day14_required_artifacts["evaluation_protocol"], "r") as f:
    evaluation_protocol = load_artifact_json(f)

with open(day14_required_artifacts["final_boundary_leakage_audit"], "r") as f:
    boundary_audit = load_artifact_json(f)

future_window_manifest = pd.read_csv(
    day14_required_artifacts["future_window_manifest"]
)

source_composition_by_split = pd.read_csv(
    day14_required_artifacts["source_composition_by_split"]
)

print("=" * 96)
print("DAY 15 — FROZEN PROTOCOL LOAD CHECK")
print("=" * 96)

print("benchmark_manifest keys :", list(benchmark_manifest.keys()))
print("split_spec keys         :", list(split_spec.keys()))
print("lora_protocol keys      :", list(lora_protocol.keys()))
print("evaluation_protocol keys:", list(evaluation_protocol.keys()))
print("boundary_audit keys     :", list(boundary_audit.keys()))

print()
print("future_window_manifest shape     :", future_window_manifest.shape)
print("source_composition_by_split shape:", source_composition_by_split.shape)

print("=" * 96)

In [ ]:
# Formal frozen-protocol integrity gate

import pandas as pd

# Frozen expected values
EXPECTED_TRAIN_N = 13_539
EXPECTED_VALID_N = 5_859
EXPECTED_FUTURE_N = 20_002

EXPECTED_VALIDATION_BOUNDARY = pd.Timestamp(
    "2025-11-01T00:00:00+00:00"
)

EXPECTED_FUTURE_CUTOFF = pd.Timestamp(
    "2026-01-01T00:00:00+00:00"
)

EXPECTED_MAX_STEPS = 200
EXPECTED_SELECTION_METRIC = "eval_loss"
EXPECTED_SELECTION_DATASET = "historical_validation only"

# Extract frozen Day 14 values

train_spec = split_spec["historical_train"]
valid_spec = split_spec["historical_validation"]
future_spec = split_spec["strictly_future_total"]

train_start = pd.Timestamp(train_spec["start_timestamp_utc"])
train_end = pd.Timestamp(train_spec["end_timestamp_utc"])

valid_start = pd.Timestamp(valid_spec["start_timestamp_utc"])
valid_end = pd.Timestamp(valid_spec["end_timestamp_utc"])

future_start = pd.Timestamp(future_spec["start_timestamp_utc"])
future_end = pd.Timestamp(future_spec["end_timestamp_utc"])

cutoff = pd.Timestamp(
    split_spec["historical_cutoff"]["cutoff_timestamp_utc"]
)

boundary_tests = {
    x["boundary_name"]: x
    for x in boundary_audit["frozen_boundary_tests"]
}

# Integrity checks

checks = {}

checks["benchmark_population_n_39400"] = (
    benchmark_manifest["benchmark_population"]["n_observations"] == 39_400
)

checks["historical_train_n_13539"] = (
    train_spec["n_observations"] == EXPECTED_TRAIN_N
)

checks["historical_validation_n_5859"] = (
    valid_spec["n_observations"] == EXPECTED_VALID_N
)

checks["strictly_future_n_20002"] = (
    future_spec["n_observations"] == EXPECTED_FUTURE_N
)

checks["population_partition_complete"] = (
    train_spec["n_observations"]
    + valid_spec["n_observations"]
    + future_spec["n_observations"]
    == benchmark_manifest["benchmark_population"]["n_observations"]
)

checks["historical_cutoff_2026_01_01"] = (
    cutoff == EXPECTED_FUTURE_CUTOFF
)

checks["validation_after_2025_11_01_boundary"] = (
    valid_start > EXPECTED_VALIDATION_BOUNDARY
)

checks["future_after_2026_01_01_cutoff"] = (
    future_start > EXPECTED_FUTURE_CUTOFF
)

checks["train_precedes_validation"] = (
    train_end < valid_start
)

checks["validation_precedes_future"] = (
    valid_end < future_start
)

checks["split_ordering_rule_frozen"] = (
    split_spec["ordering_rule"]
    == "historical_train < historical_validation < all future evaluation windows"
)

checks["split_spec_locked"] = (
    split_spec["freeze_status"] == "LOCKED"
)

checks["training_budget_200_steps"] = (
    lora_protocol["training"]["max_steps"] == EXPECTED_MAX_STEPS
)

checks["checkpoint_metric_eval_loss"] = (
    lora_protocol["checkpoint_selection"]["selection_metric"]
    == EXPECTED_SELECTION_METRIC
)

checks["checkpoint_dataset_historical_validation_only"] = (
    lora_protocol["checkpoint_selection"]["selection_dataset"]
    == EXPECTED_SELECTION_DATASET
)

checks["future_metrics_forbidden_for_selection"] = (
    lora_protocol["checkpoint_selection"]["future_window_metrics_allowed"]
    is False
)

checks["train_validation_boundary_gate_pass"] = (
    boundary_tests["train_to_validation"]["gate"] == "PASS"
    and boundary_tests["train_to_validation"]["cross_boundary_suspicious_rows"] == 0
)

checks["development_future_boundary_gate_pass"] = (
    boundary_tests["development_to_future"]["gate"] == "PASS"
    and boundary_tests["development_to_future"]["cross_boundary_suspicious_rows"] == 0
)

checks["final_boundary_leakage_gate_pass"] = (
    boundary_audit["day14_final_boundary_leakage_gate"] == "PASS"
)

checks["six_future_windows_frozen"] = (
    len(split_spec["future_windows"]) == 6
    and [x["future_window"] for x in split_spec["future_windows"]]
    == ["F1", "F2", "F3", "F4", "F5", "F6"]
)

checks["F6_partial_month_retained"] = (
    split_spec["future_windows"][-1]["future_window"] == "F6"
    and split_spec["future_windows"][-1]["is_partial_month"] is True
)

print("=" * 100)
print("DAY 15 — FORMAL FROZEN-PROTOCOL INTEGRITY GATE")
print("=" * 100)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL':<7} — {name}")

print("-" * 100)

protocol_integrity_pass = all(checks.values())

print(
    "DAY 15 PROTOCOL INTEGRITY:",
    "PASS" if protocol_integrity_pass else "FAIL"
)

print()
print(f"Historical Train      : {train_spec['n_observations']:,}")
print(f"Historical Validation : {valid_spec['n_observations']:,}")
print(f"Strictly Future       : {future_spec['n_observations']:,}")
print(f"Training budget       : {lora_protocol['training']['max_steps']} steps")
print(
    "Checkpoint rule      : "
    f"minimize {lora_protocol['checkpoint_selection']['selection_metric']} "
    f"on {lora_protocol['checkpoint_selection']['selection_dataset']}"
)
print("Future metrics allowed:",
      lora_protocol["checkpoint_selection"]["future_window_metrics_allowed"])

print("=" * 100)

assert protocol_integrity_pass, (
    "DAY 15 STOP: frozen Day 14 protocol integrity check failed."
)

In [ ]:

# Load and inspect the Day 07 row-level Dataset B source

import pandas as pd

ROW_LEVEL_DATA_PATH = (
    PROJECT_ROOT
    / "results"
    / "day07"
    / "temporal_dataset_clean.csv"
)

assert ROW_LEVEL_DATA_PATH.exists(), (
    f"Missing row-level source file: {ROW_LEVEL_DATA_PATH}"
)

dataset_b = pd.read_csv(ROW_LEVEL_DATA_PATH)

print("=" * 110)
print("DAY 15 — ROW-LEVEL DATASET B SOURCE INSPECTION")
print("=" * 110)

print(f"Source file : {ROW_LEVEL_DATA_PATH}")
print(f"Shape       : {dataset_b.shape}")

print("\nColumns:")
for i, col in enumerate(dataset_b.columns, start=1):
    print(f"{i:>2}. {col}")

print("\nFirst 3 rows:")
display(dataset_b.head(3))

print("=" * 110)

In [ ]:
# Materialize the exact frozen Day 14 benchmark population

import pandas as pd

# Parse timestamp

dataset_b["date"] = pd.to_datetime(
    dataset_b["date"],
    utc=True,
    errors="raise"
)

# Frozen benchmark boundaries from Day 14

benchmark_start = pd.Timestamp(
    benchmark_manifest["benchmark_population"]["start_timestamp_utc"]
)

benchmark_end = pd.Timestamp(
    benchmark_manifest["benchmark_population"]["end_timestamp_utc"]
)

expected_benchmark_n = benchmark_manifest[
    "benchmark_population"
]["n_observations"]

# Apply the exact frozen population rule

benchmark_population = (
    dataset_b.loc[
        (dataset_b["date"] >= benchmark_start)
        & (dataset_b["date"] <= benchmark_end)
    ]
    .copy()
    .sort_values(["date", "message_id"])
    .reset_index(drop=True)
)

excluded_population = dataset_b.loc[
    ~dataset_b.index.isin(benchmark_population.index)
].copy()

# NOTE:
# The excluded_population object above is not used downstream.
# It is only for reporting the raw-count difference.
# We calculate its size directly below from total counts.

excluded_n = len(dataset_b) - len(benchmark_population)

checks = {
    "benchmark_n_39400":
        len(benchmark_population) == expected_benchmark_n == 39_400,

    "benchmark_start_exact":
        benchmark_population["date"].min() == benchmark_start,

    "benchmark_end_exact":
        benchmark_population["date"].max() == benchmark_end,

    "chronologically_sorted":
        benchmark_population["date"].is_monotonic_increasing,

    "excluded_sparse_rows_565":
        excluded_n == 565,

    "message_id_nonmissing":
        benchmark_population["message_id"].notna().all(),

    "timestamp_nonmissing":
        benchmark_population["date"].notna().all(),

    "text_column_present":
        "text" in benchmark_population.columns,

    "sentiment_column_present":
        "sentiment" in benchmark_population.columns,

    "source_column_present":
        "channel_username" in benchmark_population.columns,
}

print("=" * 104)
print("DAY 15 — FROZEN BENCHMARK POPULATION MATERIALIZATION")
print("=" * 104)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL':<7} — {name}")

print("-" * 104)

benchmark_population_pass = all(checks.values())

print(
    "BENCHMARK POPULATION CHECKPOINT:",
    "PASS" if benchmark_population_pass else "FAIL"
)

print()
print(f"Raw Dataset B rows     : {len(dataset_b):,}")
print(f"Excluded sparse rows   : {excluded_n:,}")
print(f"Frozen benchmark rows  : {len(benchmark_population):,}")
print()
print(
    "Benchmark start       :",
    benchmark_population["date"].min()
)
print(
    "Benchmark end         :",
    benchmark_population["date"].max()
)

print("=" * 104)

assert benchmark_population_pass, (
    "DAY 15 STOP: reconstructed benchmark population "
    "does not exactly match the Day 14 frozen population."
)

In [ ]:
# Materialize frozen Historical Train and Historical Validation
# Keep future observations sealed
# Observation identity = (channel_username, message_id)

# Frozen boundaries from Day 14

train_start = pd.Timestamp(
    split_spec["historical_train"]["start_timestamp_utc"]
)
train_end = pd.Timestamp(
    split_spec["historical_train"]["end_timestamp_utc"]
)

valid_start = pd.Timestamp(
    split_spec["historical_validation"]["start_timestamp_utc"]
)
valid_end = pd.Timestamp(
    split_spec["historical_validation"]["end_timestamp_utc"]
)

future_start = pd.Timestamp(
    split_spec["strictly_future_total"]["start_timestamp_utc"]
)
# Materialize ONLY Historical Train and Validation

historical_train = (
    benchmark_population.loc[
        (benchmark_population["date"] >= train_start)
        & (benchmark_population["date"] <= train_end)
    ]
    .copy()
    .reset_index(drop=True)
)

historical_validation = (
    benchmark_population.loc[
        (benchmark_population["date"] >= valid_start)
        & (benchmark_population["date"] <= valid_end)
    ]
    .copy()
    .reset_index(drop=True)
)

# Stable observation identity
# message_id alone is NOT globally unique across channels.
# Use (channel_username, message_id) as the observation key.

historical_train["observation_id"] = (
    historical_train["channel_username"].astype(str)
    + "::"
    + historical_train["message_id"].astype(str)
)

historical_validation["observation_id"] = (
    historical_validation["channel_username"].astype(str)
    + "::"
    + historical_validation["message_id"].astype(str)
)

train_obs_ids = set(historical_train["observation_id"])
valid_obs_ids = set(historical_validation["observation_id"])

# Future remains sealed:
# count only; no future dataframe is materialized

future_mask = benchmark_population["date"] >= future_start
future_n_only = int(future_mask.sum())

checks = {
    "historical_train_n_13539":
        len(historical_train) == 13_539,

    "historical_validation_n_5859":
        len(historical_validation) == 5_859,

    "future_n_20002_count_only":
        future_n_only == 20_002,

    "train_start_exact":
        historical_train["date"].min() == train_start,

    "train_end_exact":
        historical_train["date"].max() == train_end,

    "validation_start_exact":
        historical_validation["date"].min() == valid_start,

    "validation_end_exact":
        historical_validation["date"].max() == valid_end,

    "train_observation_id_unique":
        historical_train["observation_id"].nunique()
        == len(historical_train),

    "validation_observation_id_unique":
        historical_validation["observation_id"].nunique()
        == len(historical_validation),

    "train_validation_no_observation_overlap":
        train_obs_ids.isdisjoint(valid_obs_ids),

    "train_precedes_validation":
        historical_train["date"].max()
        < historical_validation["date"].min(),

    "validation_precedes_future":
        historical_validation["date"].max()
        < future_start,

    "train_chronologically_sorted":
        historical_train["date"].is_monotonic_increasing,

    "validation_chronologically_sorted":
        historical_validation["date"].is_monotonic_increasing,

    "future_dataframe_not_materialized":
        "future_data" not in globals()
        and "future_df" not in globals()
        and "future_dataset" not in globals(),
}

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------
print("=" * 108)
print("DAY 15 — HISTORICAL TRAIN / VALIDATION MATERIALIZATION")
print("=" * 108)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL':<7} — {name}")

print("-" * 108)

historical_split_pass = all(checks.values())

print(
    "HISTORICAL SPLIT CHECKPOINT:",
    "PASS" if historical_split_pass else "FAIL"
)

print()
print(f"Historical Train      : {len(historical_train):,}")
print(f"Historical Validation : {len(historical_validation):,}")
print(f"Future count only     : {future_n_only:,}")
print()

print(
    "Train range          :",
    historical_train["date"].min(),
    "→",
    historical_train["date"].max()
)

print(
    "Validation range     :",
    historical_validation["date"].min(),
    "→",
    historical_validation["date"].max()
)

print()
print(
    "Observation ID rule  : "
    "(channel_username, message_id)"
)
print(
    "Train/Validation overlap:",
    len(train_obs_ids & valid_obs_ids)
)
print("Future dataframe created: False")

print("=" * 108)

assert historical_split_pass, (
    "DAY 15 STOP: historical Train/Validation materialization "
    "does not match the frozen Day 14 split."
)

In [ ]:
# Pre-training data sanity audit

from collections import Counter

ALLOWED_LABELS = {"negative", "neutral", "positive"}

# Helper summaries
train_label_counts = (
    historical_train["sentiment"]
    .value_counts()
    .reindex(["negative", "neutral", "positive"], fill_value=0)
)

valid_label_counts = (
    historical_validation["sentiment"]
    .value_counts()
    .reindex(["negative", "neutral", "positive"], fill_value=0)
)

train_source_counts = (
    historical_train["channel_username"]
    .value_counts()
    .sort_index()
)

valid_source_counts = (
    historical_validation["channel_username"]
    .value_counts()
    .sort_index()
)
# Missing / blank input checks

train_blank_text = (
    historical_train["text"].isna()
    | historical_train["text"].astype(str).str.strip().eq("")
)

valid_blank_text = (
    historical_validation["text"].isna()
    | historical_validation["text"].astype(str).str.strip().eq("")
)
# Label-domain checks
train_labels = set(
    historical_train["sentiment"].dropna().astype(str).unique()
)

valid_labels = set(
    historical_validation["sentiment"].dropna().astype(str).unique()
)

# Compare class counts against frozen Day 14 spec
expected_train_counts = {
    "negative": split_spec["historical_train"]["negative_n"],
    "neutral": split_spec["historical_train"]["neutral_n"],
    "positive": split_spec["historical_train"]["positive_n"],
}

expected_valid_counts = {
    "negative": split_spec["historical_validation"]["negative_n"],
    "neutral": split_spec["historical_validation"]["neutral_n"],
    "positive": split_spec["historical_validation"]["positive_n"],
}
# Formal audit checks
checks = {
    "train_no_missing_or_blank_text":
        int(train_blank_text.sum()) == 0,

    "validation_no_missing_or_blank_text":
        int(valid_blank_text.sum()) == 0,

    "train_labels_valid":
        train_labels == ALLOWED_LABELS,

    "validation_labels_valid":
        valid_labels == ALLOWED_LABELS,

    "train_no_missing_labels":
        historical_train["sentiment"].notna().all(),

    "validation_no_missing_labels":
        historical_validation["sentiment"].notna().all(),

    "train_no_missing_source":
        historical_train["channel_username"].notna().all(),

    "validation_no_missing_source":
        historical_validation["channel_username"].notna().all(),

    "train_no_missing_observation_id":
        historical_train["observation_id"].notna().all(),

    "validation_no_missing_observation_id":
        historical_validation["observation_id"].notna().all(),

    "train_observation_ids_unique":
        historical_train["observation_id"].nunique()
        == len(historical_train),

    "validation_observation_ids_unique":
        historical_validation["observation_id"].nunique()
        == len(historical_validation),

    "train_validation_no_overlap":
        set(historical_train["observation_id"]).isdisjoint(
            set(historical_validation["observation_id"])
        ),

    "train_label_counts_match_day14":
        train_label_counts.to_dict() == expected_train_counts,

    "validation_label_counts_match_day14":
        valid_label_counts.to_dict() == expected_valid_counts,

    "train_source_count_matches_day14":
        historical_train["channel_username"].nunique()
        == split_spec["historical_train"]["n_sources"],

    "validation_source_count_matches_day14":
        historical_validation["channel_username"].nunique()
        == split_spec["historical_validation"]["n_sources"],

    "train_dates_within_frozen_range":
        historical_train["date"].min() == train_start
        and historical_train["date"].max() == train_end,

    "validation_dates_within_frozen_range":
        historical_validation["date"].min() == valid_start
        and historical_validation["date"].max() == valid_end,

    "future_observations_in_train_zero":
        int(
            (
                historical_train["date"]
                >= pd.Timestamp("2026-01-01T00:00:00+00:00")
            ).sum()
        ) == 0,

    "future_observations_in_validation_zero":
        int(
            (
                historical_validation["date"]
                >= pd.Timestamp("2026-01-01T00:00:00+00:00")
            ).sum()
        ) == 0,
}

print("=" * 108)
print("DAY 15 — PRE-TRAINING DATA SANITY AUDIT")
print("=" * 108)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL':<7} — {name}")

print("-" * 108)

data_sanity_pass = all(checks.values())

print(
    "PRE-TRAINING DATA SANITY:",
    "PASS" if data_sanity_pass else "FAIL"
)

print("\nHistorical Train label counts:")
for label, n in train_label_counts.items():
    print(f"  {label:<8}: {int(n):,}")

print("\nHistorical Validation label counts:")
for label, n in valid_label_counts.items():
    print(f"  {label:<8}: {int(n):,}")

print()
print(
    "Historical Train sources      :",
    historical_train["channel_username"].nunique()
)
print(
    "Historical Validation sources :",
    historical_validation["channel_username"].nunique()
)

print()
print(
    "Blank Train inputs            :",
    int(train_blank_text.sum())
)
print(
    "Blank Validation inputs       :",
    int(valid_blank_text.sum())
)

print()
print(
    "Future observations in Train  :",
    int(
        (
            historical_train["date"]
            >= pd.Timestamp("2026-01-01T00:00:00+00:00")
        ).sum()
    )
)
print(
    "Future observations in Valid  :",
    int(
        (
            historical_validation["date"]
            >= pd.Timestamp("2026-01-01T00:00:00+00:00")
        ).sum()
    )
)

print("=" * 108)

assert data_sanity_pass, (
    "DAY 15 STOP: pre-training data sanity audit failed."
)

In [ ]:
# Save historical Train/Validation manifests and data audit
import json
from datetime import datetime, timezone

# Helper: source composition
def source_composition(df):
    counts = (
        df["channel_username"]
        .value_counts()
        .sort_index()
    )

    total = len(df)

    return {
        source: {
            "n": int(n),
            "proportion": float(n / total),
        }
        for source, n in counts.items()
    }
# Historical Train manifest
historical_train_manifest = {
    "day": 15,
    "artifact": "historical_train_manifest",
    "project": "Financial Language Models under Temporal Distribution Shift",

    "split_name": "historical_train",
    "dataset": "Dataset B",

    "n_observations": int(len(historical_train)),
    "n_sources": int(
        historical_train["channel_username"].nunique()
    ),

    "start_timestamp_utc": historical_train["date"].min().isoformat(),
    "end_timestamp_utc": historical_train["date"].max().isoformat(),

    "label_counts": {
        label: int(n)
        for label, n in train_label_counts.items()
    },

    "label_proportions": {
        label: float(n / len(historical_train))
        for label, n in train_label_counts.items()
    },

    "source_composition": source_composition(
        historical_train
    ),

    "observation_id_rule": (
        "channel_username::message_id"
    ),

    "observation_id_unique": bool(
        historical_train["observation_id"].nunique()
        == len(historical_train)
    ),

    "day14_protocol_reference": {
        "chronological_split_spec":
            str(
                day14_required_artifacts[
                    "chronological_split_spec"
                ]
            ),
        "benchmark_population_manifest":
            str(
                day14_required_artifacts[
                    "benchmark_population_manifest"
                ]
            ),
    },

    "future_observations_in_split": 0,

    "freeze_status": "LOCKED_FOR_DAY15_TRAINING",
}

# Historical Validation manifest

historical_validation_manifest = {
    "day": 15,
    "artifact": "historical_validation_manifest",
    "project": "Financial Language Models under Temporal Distribution Shift",

    "split_name": "historical_validation",
    "dataset": "Dataset B",

    "n_observations": int(len(historical_validation)),
    "n_sources": int(
        historical_validation["channel_username"].nunique()
    ),

    "start_timestamp_utc":
        historical_validation["date"].min().isoformat(),

    "end_timestamp_utc":
        historical_validation["date"].max().isoformat(),

    "label_counts": {
        label: int(n)
        for label, n in valid_label_counts.items()
    },

    "label_proportions": {
        label: float(n / len(historical_validation))
        for label, n in valid_label_counts.items()
    },

    "source_composition": source_composition(
        historical_validation
    ),

    "observation_id_rule": (
        "channel_username::message_id"
    ),

    "observation_id_unique": bool(
        historical_validation["observation_id"].nunique()
        == len(historical_validation)
    ),

    "train_validation_overlap": 0,

    "day14_protocol_reference": {
        "chronological_split_spec":
            str(
                day14_required_artifacts[
                    "chronological_split_spec"
                ]
            ),
        "benchmark_population_manifest":
            str(
                day14_required_artifacts[
                    "benchmark_population_manifest"
                ]
            ),
    },

    "future_observations_in_split": 0,

    "freeze_status": "LOCKED_FOR_DAY15_VALIDATION",
}

# Training data audit

training_data_audit = {
    "day": 15,
    "artifact": "training_data_audit",

    "audit_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "historical_train_n":
        int(len(historical_train)),

    "historical_validation_n":
        int(len(historical_validation)),

    "future_n_verified_count_only":
        int(future_n_only),

    "future_dataframe_materialized":
        False,

    "future_observations_used_for_training":
        0,

    "future_observations_used_for_validation":
        0,

    "observation_id_rule":
        "channel_username::message_id",

    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },

    "all_checks_pass":
        bool(data_sanity_pass),

    "train_validation_overlap":
        int(
            len(
                set(historical_train["observation_id"])
                &
                set(historical_validation["observation_id"])
            )
        ),

    "chronology": {
        "train_end_utc":
            historical_train["date"].max().isoformat(),

        "validation_start_utc":
            historical_validation["date"].min().isoformat(),

        "validation_end_utc":
            historical_validation["date"].max().isoformat(),

        "future_start_utc":
            split_spec[
                "strictly_future_total"
            ]["start_timestamp_utc"],
    },

    "protocol_status":
        "PASS",
}

train_manifest_path = (
    DAY15_RESULT_DIR
    / "historical_train_manifest.json"
)

validation_manifest_path = (
    DAY15_RESULT_DIR
    / "historical_validation_manifest.json"
)

training_audit_path = (
    DAY15_RESULT_DIR
    / "training_data_audit.json"
)


with open(train_manifest_path, "w") as f:
    json.dump(
        historical_train_manifest,
        f,
        indent=2
    )

with open(validation_manifest_path, "w") as f:
    json.dump(
        historical_validation_manifest,
        f,
        indent=2
    )

with open(training_audit_path, "w") as f:
    json.dump(
        training_data_audit,
        f,
        indent=2
    )

# Artifact checkpoint

artifact_paths = {
    "historical_train_manifest.json":
        train_manifest_path,

    "historical_validation_manifest.json":
        validation_manifest_path,

    "training_data_audit.json":
        training_audit_path,
}


print("=" * 108)
print("DAY 15 — DATA-LAYER ARTIFACT CHECKPOINT")
print("=" * 108)

for name, path in artifact_paths.items():
    exists = path.exists()
    size = path.stat().st_size if exists else 0

    print(
        f"{'PASS' if exists else 'FAIL':<7} — "
        f"{name:<40} "
        f"{size:>8,} bytes"
    )

print("-" * 108)

data_artifacts_pass = all(
    path.exists()
    for path in artifact_paths.values()
)

print(
    "DATA-LAYER ARTIFACT STATUS:",
    "PASS" if data_artifacts_pass else "FAIL"
)

print()
print("Saved to:")
print(DAY15_RESULT_DIR)

print("=" * 108)

assert data_artifacts_pass, (
    "DAY 15 STOP: one or more data-layer artifacts "
    "were not saved."
)

## Stage D — Fresh Historical Dataset B LoRA Initialization

Day 15 now initializes a new LoRA model, denoted \(M_H\), for the primary within-Dataset-B temporal benchmark.

\(M_H\) is **not** a continuation of the Day 04 Dataset-A-trained adapter.

The two models serve different scientific roles:

- **\(M_A\)** — trained on Dataset A  
  External OOD / cross-dataset robustness baseline.

- **\(M_H\)** — trained on Historical Dataset B  
  Primary model for future temporal generalization evaluation.

Both models share the same base-model family and comparable LoRA architecture, but the supervised training data are kept strictly separate.

For \(M_H\):

\[
\text{Historical Train}
\rightarrow
\text{fresh LoRA adapter}
\rightarrow
\text{200-step training}
\rightarrow
\text{historical validation-loss checkpoint selection}
\rightarrow
\text{model freeze}
\]

No future F1–F6 observation or metric is used during this stage.

In [ ]:

# Runtime and package environment check
import sys
import torch
import transformers
import datasets
import peft
import accelerate

print("=" * 108)
print("DAY 15 — RUNTIME ENVIRONMENT CHECK")
print("=" * 108)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"Accelerate   : {accelerate.__version__}")

print()
print(f"CUDA available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU capability : "
        f"{torch.cuda.get_device_capability(0)}"
    )
    print(
        f"CUDA runtime   : "
        f"{torch.version.cuda}"
    )

    total_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        f"GPU memory     : "
        f"{total_memory_gb:.2f} GB"
    )

print("-" * 108)

runtime_pass = torch.cuda.is_available()

print(
    "GPU RUNTIME CHECKPOINT:",
    "PASS" if runtime_pass else "FAIL"
)

print("=" * 108)

assert runtime_pass, (
    "DAY 15 STOP: CUDA GPU is required before loading "
    "Meta-Llama-3-8B for LoRA training."
)

In [ ]:
# ============================================================
# DAY 15 — CELL 12
# Formal frozen model-configuration integrity gate
# ============================================================

# ------------------------------------------------------------
# Extract frozen protocol blocks
# ------------------------------------------------------------
base_model_cfg = lora_protocol["base_model"]
quant_cfg = lora_protocol["quantization"]
lora_cfg = lora_protocol["lora"]
seq_cfg = lora_protocol["sequence_protocol"]
train_cfg = lora_protocol["training"]
ckpt_cfg = lora_protocol["checkpoint_selection"]
infer_cfg = lora_protocol["inference"]
comparability_cfg = lora_protocol["day04_comparability"]

# ------------------------------------------------------------
# Formal checks
# ------------------------------------------------------------
model_protocol_checks = {
    # Base model
    "base_model_meta_llama_3_8b":
        base_model_cfg["model_name"]
        == "meta-llama/Meta-Llama-3-8B",

    "task_causal_language_modeling":
        base_model_cfg["task"]
        == "causal_language_modeling",

    # Quantization
    "quantization_enabled":
        quant_cfg["enabled"] is True,

    "load_in_4bit":
        quant_cfg["load_in_4bit"] is True,

    "quant_type_nf4":
        quant_cfg["bnb_4bit_quant_type"] == "nf4",

    "double_quant_enabled":
        quant_cfg["bnb_4bit_use_double_quant"] is True,

    "compute_dtype_float16":
        quant_cfg["bnb_4bit_compute_dtype"] == "float16",

    # LoRA
    "lora_r_8":
        lora_cfg["r"] == 8,

    "lora_alpha_32":
        lora_cfg["lora_alpha"] == 32,

    "lora_dropout_0_1":
        lora_cfg["lora_dropout"] == 0.1,

    "lora_target_modules_q_v":
        lora_cfg["target_modules"] == ["q_proj", "v_proj"],

    "lora_task_type_causal_lm":
        lora_cfg["task_type"] == "CAUSAL_LM",

    # Sequence / prompt
    "max_seq_length_512":
        seq_cfg["max_seq_length"] == 512,

    "prompt_template_frozen":
        seq_cfg["prompt_template"]
        == r"Instruction:{instruction}\nInput:{input}\nAnswer: ",

    "target_labels_frozen":
        seq_cfg["target_labels"]
        == ["negative", "neutral", "positive"],

    "append_eos":
        seq_cfg["append_eos_token"] is True,

    "loss_masking_frozen":
        seq_cfg["padding_label_value"] == -100,

    # Training
    "training_budget_200":
        train_cfg["max_steps"] == 200,

    "learning_rate_1e_4":
        train_cfg["learning_rate"] == 1e-4,

    "train_batch_size_1":
        train_cfg["per_device_train_batch_size"] == 1,

    "grad_accumulation_8":
        train_cfg["gradient_accumulation_steps"] == 8,

    "effective_batch_size_8":
        train_cfg["effective_train_batch_size"] == 8,

    "seed_42":
        train_cfg["seed"] == 42,

    "data_seed_42":
        train_cfg["data_seed"] == 42,

    "eval_steps_25":
        train_cfg["eval_steps"] == 25,

    "save_steps_50":
        train_cfg["save_steps"] == 50,

    # Checkpoint rule
    "checkpoint_metric_eval_loss":
        ckpt_cfg["selection_metric"] == "eval_loss",

    "checkpoint_direction_minimize":
        ckpt_cfg["selection_direction"] == "minimize",

    "checkpoint_dataset_historical_validation_only":
        ckpt_cfg["selection_dataset"]
        == "historical_validation only",

    "candidate_steps_frozen":
        ckpt_cfg["candidate_steps"]
        == [50, 100, 150, 200],

    "early_stopping_disabled":
        ckpt_cfg["early_stopping"] is False,

    "future_metrics_forbidden":
        ckpt_cfg["future_window_metrics_allowed"] is False,

    # Inference
    "inference_eval_mode":
        infer_cfg["model_mode"] == "eval",

    "greedy_generation":
        infer_cfg["generation"]["do_sample"] is False,

    "max_new_tokens_4":
        infer_cfg["generation"]["max_new_tokens"] == 4,

    "valid_predictions_frozen":
        infer_cfg["valid_predictions"]
        == ["negative", "neutral", "positive"],

    # Day 04 comparability
    "architecture_comparable_to_day04":
        comparability_cfg["core_architecture_preserved"] is True,

    "lora_comparable_to_day04":
        comparability_cfg["lora_configuration_preserved"] is True,

    "optimization_comparable_to_day04":
        comparability_cfg["optimization_hyperparameters_preserved"] is True,

    "quantization_comparable_to_day04":
        comparability_cfg["quantization_preserved"] is True,

    "sequence_length_comparable_to_day04":
        comparability_cfg["sequence_length_preserved"] is True,

    # Frozen before training
    "protocol_locked_before_day15":
        lora_protocol["freeze_status"]
        == "LOCKED_BEFORE_DAY15",
}

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------
print("=" * 110)
print("DAY 15 — FORMAL MODEL CONFIGURATION INTEGRITY GATE")
print("=" * 110)

for name, passed in model_protocol_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<7} — {name}")

print("-" * 110)

model_protocol_pass = all(model_protocol_checks.values())

print(
    "MODEL CONFIGURATION INTEGRITY:",
    "PASS" if model_protocol_pass else "FAIL"
)

print()
print(f"Base model       : {base_model_cfg['model_name']}")

print(
    "Quantization    : "
    "4-bit NF4, double quant, float16 compute"
)

print(
    "LoRA            : "
    f"r={lora_cfg['r']}, "
    f"alpha={lora_cfg['lora_alpha']}, "
    f"dropout={lora_cfg['lora_dropout']}, "
    f"targets={lora_cfg['target_modules']}"
)

print(
    f"Max seq length   : "
    f"{seq_cfg['max_seq_length']}"
)

print(
    f"Training budget  : "
    f"{train_cfg['max_steps']} steps"
)

print(
    "Checkpoint rule : "
    f"{ckpt_cfg['selection_direction']} "
    f"{ckpt_cfg['selection_metric']} "
    f"on {ckpt_cfg['selection_dataset']}"
)

print(
    "Candidate steps :",
    ckpt_cfg["candidate_steps"]
)

print(
    "Future metrics  :",
    ckpt_cfg["future_window_metrics_allowed"]
)

print(
    "Prompt template :",
    repr(seq_cfg["prompt_template"])
)

print("=" * 110)

assert model_protocol_pass, (
    "DAY 15 STOP: frozen model configuration "
    "does not match the Day 14 protocol."
)

In [ ]:
# Load tokenizer and define frozen prompt / target encoding
from transformers import AutoTokenizer

# Load tokenizer from the frozen base model

tokenizer = AutoTokenizer.from_pretrained(
    base_model_cfg["model_name"],
    trust_remote_code=base_model_cfg["trust_remote_code"],
)

# Frozen training convention
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_SEQ_LENGTH = seq_cfg["max_seq_length"]
PROMPT_TEMPLATE = seq_cfg["prompt_template"]
VALID_LABELS = set(seq_cfg["target_labels"])

# Fixed instruction used for sentiment classification
INSTRUCTION_TEXT = (
    "Classify the sentiment of the following financial news "
    "as negative, neutral, or positive."
)

# Encoding function

def encode_training_example(text, label):
    """
    Build one causal-LM training example under the frozen Day 15 protocol.

    Loss is masked over the complete prompt.
    Only target label + EOS contribute to training loss.
    """

    text = str(text)
    label = str(label).strip().lower()

    if label not in VALID_LABELS:
        raise ValueError(f"Invalid sentiment label: {label}")

    # IMPORTANT:
    # PROMPT_TEMPLATE contains literal backslash-n sequences
    # because this is the exact Day 14 frozen representation.
    prompt = PROMPT_TEMPLATE.format(
        instruction=INSTRUCTION_TEXT,
        input=text,
    )

    target = label + tokenizer.eos_token

    # Tokenize prompt and target separately so prompt loss
    # can be masked exactly.
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    # Keep target intact.
    # If truncation is required, truncate the prompt portion.
    max_prompt_length = MAX_SEQ_LENGTH - len(target_ids)

    if max_prompt_length <= 0:
        raise ValueError(
            "Target token length exceeds MAX_SEQ_LENGTH."
        )

    if len(prompt_ids) > max_prompt_length:
        prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + target_ids

    attention_mask = [1] * len(input_ids)

    labels = (
        [-100] * len(prompt_ids)
        + target_ids.copy()
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "prompt_token_count": len(prompt_ids),
        "target_token_count": len(target_ids),
        "sequence_length": len(input_ids),
        "prompt_text": prompt,
        "target_text": target,
    }

# Basic tokenizer / encoding checks

tokenizer_checks = {
    "tokenizer_loaded":
        tokenizer is not None,

    "pad_token_is_eos":
        tokenizer.pad_token_id == tokenizer.eos_token_id,

    "training_padding_side_right":
        tokenizer.padding_side == "right",

    "max_seq_length_512":
        MAX_SEQ_LENGTH == 512,

    "valid_labels_frozen":
        VALID_LABELS == {
            "negative",
            "neutral",
            "positive"
        },

    "literal_backslash_n_preserved":
        "\\n" in PROMPT_TEMPLATE,
}

print("=" * 108)
print("DAY 15 — TOKENIZER & FROZEN ENCODING SETUP")
print("=" * 108)

for name, passed in tokenizer_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<7} — {name}")

print("-" * 108)

tokenizer_setup_pass = all(tokenizer_checks.values())

print(
    "TOKENIZER / ENCODING SETUP:",
    "PASS" if tokenizer_setup_pass else "FAIL"
)

print()
print(f"Tokenizer          : {base_model_cfg['model_name']}")
print(f"EOS token          : {repr(tokenizer.eos_token)}")
print(f"EOS token ID       : {tokenizer.eos_token_id}")
print(f"Pad token          : {repr(tokenizer.pad_token)}")
print(f"Pad token ID       : {tokenizer.pad_token_id}")
print(f"Padding side       : {tokenizer.padding_side}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

print()
print("Frozen prompt repr:")
print(repr(PROMPT_TEMPLATE))

print("=" * 108)

assert tokenizer_setup_pass, (
    "DAY 15 STOP: tokenizer or frozen encoding setup failed."
)

In [ ]:
# Build encoded Hugging Face datasets for Train / Validation
from datasets import Dataset

# Keep only the fields needed for training / validation
# plus observation metadata for auditability

train_frame = historical_train[
    [
        "observation_id",
        "channel_username",
        "message_id",
        "date",
        "text",
        "sentiment",
    ]
].copy()

valid_frame = historical_validation[
    [
        "observation_id",
        "channel_username",
        "message_id",
        "date",
        "text",
        "sentiment",
    ]
].copy()

# Datasets does not need pandas Timestamp objects
train_frame["date"] = train_frame["date"].astype(str)
valid_frame["date"] = valid_frame["date"].astype(str)

# Convert to Hugging Face Dataset

hf_train_raw = Dataset.from_pandas(
    train_frame,
    preserve_index=False,
)

hf_valid_raw = Dataset.from_pandas(
    valid_frame,
    preserve_index=False,
)

# Batch encoding

def encode_batch(batch):
    output = {
        "input_ids": [],
        "attention_mask": [],
        "labels": [],
    }

    for text, label in zip(
        batch["text"],
        batch["sentiment"],
    ):
        encoded = encode_training_example(
            text=text,
            label=label,
        )

        output["input_ids"].append(
            encoded["input_ids"]
        )

        output["attention_mask"].append(
            encoded["attention_mask"]
        )

        output["labels"].append(
            encoded["labels"]
        )

    return output

# Encode full Train / Validation datasets

hf_train = hf_train_raw.map(
    encode_batch,
    batched=True,
    desc="Encoding Historical Train",
)

hf_valid = hf_valid_raw.map(
    encode_batch,
    batched=True,
    desc="Encoding Historical Validation",
)

# Sequence-length diagnostics

train_lengths = [
    len(x)
    for x in hf_train["input_ids"]
]

valid_lengths = [
    len(x)
    for x in hf_valid["input_ids"]
]

encoded_dataset_checks = {
    "train_n_13539":
        len(hf_train) == 13_539,

    "validation_n_5859":
        len(hf_valid) == 5_859,

    "train_all_within_512":
        max(train_lengths) <= MAX_SEQ_LENGTH,

    "validation_all_within_512":
        max(valid_lengths) <= MAX_SEQ_LENGTH,

    "train_labels_present":
        "labels" in hf_train.column_names,

    "validation_labels_present":
        "labels" in hf_valid.column_names,

    "train_attention_mask_present":
        "attention_mask" in hf_train.column_names,

    "validation_attention_mask_present":
        "attention_mask" in hf_valid.column_names,

    "train_observation_id_preserved":
        "observation_id" in hf_train.column_names,

    "validation_observation_id_preserved":
        "observation_id" in hf_valid.column_names,
}

print("=" * 108)
print("DAY 15 — ENCODED HISTORICAL DATASET CHECKPOINT")
print("=" * 108)

for name, passed in encoded_dataset_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 108)

encoded_dataset_pass = all(
    encoded_dataset_checks.values()
)

print(
    "ENCODED DATASET STATUS:",
    "PASS" if encoded_dataset_pass else "FAIL"
)

print()
print(f"Historical Train      : {len(hf_train):,}")
print(f"Historical Validation : {len(hf_valid):,}")

print()
print(
    "Train sequence length      : "
    f"min={min(train_lengths)}, "
    f"max={max(train_lengths)}, "
    f"mean={sum(train_lengths)/len(train_lengths):.2f}"
)

print(
    "Validation sequence length : "
    f"min={min(valid_lengths)}, "
    f"max={max(valid_lengths)}, "
    f"mean={sum(valid_lengths)/len(valid_lengths):.2f}"
)

print()
print(
    "Train columns:",
    hf_train.column_names
)

print(
    "Validation columns:",
    hf_valid.column_names
)

print("=" * 108)

assert encoded_dataset_pass, (
    "DAY 15 STOP: encoded historical datasets failed "
    "the frozen-protocol checks."
)

In [ ]:
!pip install -q -U bitsandbytes

import bitsandbytes as bnb

print("=" * 88)
print("DAY 15 — BITSANDBYTES DEPENDENCY CHECK")
print("=" * 88)
print("bitsandbytes version :", bnb.__version__)
print("Installation status  : PASS")
print("=" * 88)

In [ ]:
# Load frozen 4-bit base model and initialize fresh M_H LoRA
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# 1. Frozen 4-bit quantization configuration

bnb_config = BitsAndBytesConfig(
    load_in_4bit=quant_cfg["load_in_4bit"],
    bnb_4bit_quant_type=quant_cfg["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=quant_cfg[
        "bnb_4bit_use_double_quant"
    ],
    bnb_4bit_compute_dtype=torch.float16,
)

# 2. Load COMMON BASE MODEL ONLY
#
# IMPORTANT:
# No Day 04 adapter is loaded here.
# M_H starts from the same base-model family with a fresh LoRA.

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_cfg["model_name"],
    quantization_config=bnb_config,
    device_map=base_model_cfg["device_map"],
    trust_remote_code=base_model_cfg["trust_remote_code"],
)

# Frozen training behavior
base_model.config.use_cache = False

# 3. Prepare quantized model for LoRA training

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=train_cfg[
        "gradient_checkpointing"
    ],
)

# 4. Create a FRESH LoRA configuration

peft_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    bias=lora_cfg["bias"],
    target_modules=lora_cfg["target_modules"],
    task_type=lora_cfg["task_type"],
    inference_mode=lora_cfg["inference_mode"],
)

# 5. Attach fresh LoRA adapter

model = get_peft_model(
    base_model,
    peft_config,
)

# 6. Trainable parameter audit

total_params = 0
trainable_params = 0

for param in model.parameters():
    n = param.numel()
    total_params += n

    if param.requires_grad:
        trainable_params += n

trainable_pct = (
    100.0 * trainable_params / total_params
)

# 7. Structural checks

model_init_checks = {
    "model_loaded":
        model is not None,

    "peft_adapter_present":
        hasattr(model, "peft_config"),

    "fresh_adapter_named_default":
        "default" in model.peft_config,

    "base_model_matches_frozen_protocol":
        base_model_cfg["model_name"]
        == "meta-llama/Meta-Llama-3-8B",

    "lora_r_matches":
        model.peft_config["default"].r
        == lora_cfg["r"],

    "lora_alpha_matches":
        model.peft_config["default"].lora_alpha
        == lora_cfg["lora_alpha"],

    "lora_dropout_matches":
        model.peft_config["default"].lora_dropout
        == lora_cfg["lora_dropout"],

    "trainable_params_positive":
        trainable_params > 0,

    "trainable_params_less_than_total":
        trainable_params < total_params,

    "use_cache_disabled":
        model.config.use_cache is False,
}
print("=" * 110)
print("DAY 15 — FRESH HISTORICAL LoRA INITIALIZATION")
print("=" * 110)

for name, passed in model_init_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 110)

model_init_pass = all(
    model_init_checks.values()
)

print(
    "FRESH M_H INITIALIZATION:",
    "PASS" if model_init_pass else "FAIL"
)

print()
print(f"Base model       : {base_model_cfg['model_name']}")
print("Adapter identity : M_H / fresh Day 15 LoRA")
print("Day 04 adapter   : NOT LOADED")

print()
print(f"Trainable params : {trainable_params:,}")
print(f"Total params     : {total_params:,}")
print(f"Trainable %      : {trainable_pct:.6f}%")

print()
print(
    "LoRA config      : "
    f"r={model.peft_config['default'].r}, "
    f"alpha={model.peft_config['default'].lora_alpha}, "
    f"dropout={model.peft_config['default'].lora_dropout}"
)

print(
    "Target modules   :",
    lora_cfg["target_modules"]
)

print()
if torch.cuda.is_available():
    allocated_gb = (
        torch.cuda.memory_allocated()
        / (1024 ** 3)
    )

    reserved_gb = (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )

    print(
        f"GPU allocated    : {allocated_gb:.2f} GB"
    )
    print(
        f"GPU reserved     : {reserved_gb:.2f} GB"
    )

print("=" * 110)

assert model_init_pass, (
    "DAY 15 STOP: fresh historical LoRA initialization failed."
)

In [ ]:
# Build Trainer-ready datasets and frozen dynamic-padding collator
import torch

# ------------------------------------------------------------
# 1. Trainer should receive model tensors only.
#
# Keep hf_train / hf_valid with metadata for auditability.
# Create separate Trainer-only views here.
# ------------------------------------------------------------
MODEL_COLUMNS = [
    "input_ids",
    "attention_mask",
    "labels",
]

hf_train_trainer = hf_train.select_columns(
    MODEL_COLUMNS
)

hf_valid_trainer = hf_valid.select_columns(
    MODEL_COLUMNS
)

# ------------------------------------------------------------
# 2. Frozen dynamic-padding collator
#
# input_ids      -> tokenizer.pad_token_id
# attention_mask -> 0
# labels         -> -100
#
# Right padding is used during training.
# ------------------------------------------------------------
class CausalLMDynamicPaddingCollator:
    def __init__(self, tokenizer, label_pad_token_id=-100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        max_len = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for feature in features:
            input_ids = list(feature["input_ids"])
            attention_mask = list(feature["attention_mask"])
            labels = list(feature["labels"])

            pad_len = max_len - len(input_ids)

            batch_input_ids.append(
                input_ids
                + [self.tokenizer.pad_token_id] * pad_len
            )

            batch_attention_mask.append(
                attention_mask
                + [0] * pad_len
            )

            batch_labels.append(
                labels
                + [self.label_pad_token_id] * pad_len
            )

        return {
            "input_ids": torch.tensor(
                batch_input_ids,
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                batch_attention_mask,
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                batch_labels,
                dtype=torch.long,
            ),
        }


data_collator = CausalLMDynamicPaddingCollator(
    tokenizer=tokenizer,
    label_pad_token_id=seq_cfg["padding_label_value"],
)

# ------------------------------------------------------------
# 3. Construct a representative mixed-length batch
# ------------------------------------------------------------
sample_indices = [0, 1, 2, 100]

sample_features = [
    hf_train_trainer[i]
    for i in sample_indices
]

test_batch = data_collator(
    sample_features
)

original_lengths = [
    len(x["input_ids"])
    for x in sample_features
]

expected_batch_length = max(
    original_lengths
)

# ------------------------------------------------------------
# 4. Padding / masking checks
# ------------------------------------------------------------
collator_checks = {
    "trainer_train_n_13539":
        len(hf_train_trainer) == 13_539,

    "trainer_validation_n_5859":
        len(hf_valid_trainer) == 5_859,

    "trainer_columns_exact":
        hf_train_trainer.column_names
        == MODEL_COLUMNS
        and hf_valid_trainer.column_names
        == MODEL_COLUMNS,

    "input_ids_tensor":
        isinstance(
            test_batch["input_ids"],
            torch.Tensor
        ),

    "attention_mask_tensor":
        isinstance(
            test_batch["attention_mask"],
            torch.Tensor
        ),

    "labels_tensor":
        isinstance(
            test_batch["labels"],
            torch.Tensor
        ),

    "batch_size_correct":
        test_batch["input_ids"].shape[0]
        == len(sample_indices),

    "dynamic_padding_correct":
        test_batch["input_ids"].shape[1]
        == expected_batch_length,

    "all_batch_tensors_same_shape":
        test_batch["input_ids"].shape
        == test_batch["attention_mask"].shape
        == test_batch["labels"].shape,

    "label_padding_is_minus_100":
        all(
            torch.all(
                test_batch["labels"][i, length:]
                == -100
            ).item()
            for i, length in enumerate(original_lengths)
        ),

    "attention_padding_is_zero":
        all(
            torch.all(
                test_batch["attention_mask"][i, length:]
                == 0
            ).item()
            for i, length in enumerate(original_lengths)
        ),

    "input_padding_is_eos":
        all(
            torch.all(
                test_batch["input_ids"][i, length:]
                == tokenizer.eos_token_id
            ).item()
            for i, length in enumerate(original_lengths)
        ),

    "batch_within_max_seq_length":
        test_batch["input_ids"].shape[1]
        <= MAX_SEQ_LENGTH,
}

# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------
print("=" * 110)
print("DAY 15 — TRAINER DATA / DYNAMIC-PADDING COLLATOR CHECK")
print("=" * 110)

for name, passed in collator_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 110)

collator_pass = all(
    collator_checks.values()
)

print(
    "TRAINER DATA / COLLATOR STATUS:",
    "PASS" if collator_pass else "FAIL"
)

print()
print(
    f"Trainer Train      : "
    f"{len(hf_train_trainer):,}"
)
print(
    f"Trainer Validation : "
    f"{len(hf_valid_trainer):,}"
)

print()
print(
    "Original sample lengths :",
    original_lengths
)
print(
    "Padded batch shape       :",
    tuple(test_batch["input_ids"].shape)
)

print()
print(
    "Input pad token ID       :",
    tokenizer.pad_token_id
)
print(
    "Label pad value          :",
    seq_cfg["padding_label_value"]
)
print(
    "Training padding side    :",
    tokenizer.padding_side
)

print("=" * 110)

assert collator_pass, (
    "DAY 15 STOP: Trainer dataset or dynamic-padding "
    "collator failed the frozen-protocol checks."
)

In [ ]:
# ============================================================
# DAY 15 — CELL 17
# Frozen TrainingArguments + Trainer initialization
# FINAL PRE-TRAINING GATE
# ============================================================

import inspect
from transformers import TrainingArguments, Trainer

# ------------------------------------------------------------
# 1. Training output directory
# ------------------------------------------------------------
TRAINING_OUTPUT_DIR = (
    DAY15_MODEL_DIR
    / "training_checkpoints"
)

TRAINING_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Build frozen TrainingArguments
#
# Transformers API compatibility:
# newer versions use "eval_strategy";
# older versions may use "evaluation_strategy".
# ------------------------------------------------------------
training_args_signature = inspect.signature(
    TrainingArguments.__init__
).parameters

training_kwargs = {
    "output_dir":
        str(TRAINING_OUTPUT_DIR),

    # Frozen optimization budget
    "max_steps":
        train_cfg["max_steps"],

    "learning_rate":
        train_cfg["learning_rate"],

    "per_device_train_batch_size":
        train_cfg["per_device_train_batch_size"],

    "per_device_eval_batch_size":
        train_cfg["per_device_eval_batch_size"],

    "gradient_accumulation_steps":
        train_cfg["gradient_accumulation_steps"],

    "weight_decay":
        train_cfg["weight_decay"],

    "warmup_steps":
        train_cfg["warmup_steps"],

    # Frozen logging / evaluation / checkpoint schedule
    "logging_steps":
        train_cfg["logging_steps"],

    "eval_steps":
        train_cfg["eval_steps"],

    "save_steps":
        train_cfg["save_steps"],

    "save_strategy":
        train_cfg["save_strategy"],

    "save_total_limit":
        train_cfg["save_total_limit"],

    # Precision / memory
    "fp16":
        train_cfg["fp16"],

    "gradient_checkpointing":
        train_cfg["gradient_checkpointing"],

    # Reproducibility
    "seed":
        train_cfg["seed"],

    "data_seed":
        train_cfg["data_seed"],

    # Dataset behavior
    "remove_unused_columns":
        train_cfg["remove_unused_columns"],

    # Reporting
    "report_to":
        train_cfg["report_to"],

    # IMPORTANT:
    # Day 15 manually selects among checkpoints
    # 50 / 100 / 150 / 200 using validation eval_loss.
    "load_best_model_at_end":
        False,
}

# ------------------------------------------------------------
# 3. Add evaluation-strategy argument using the name
# supported by the current Transformers version
# ------------------------------------------------------------
if "eval_strategy" in training_args_signature:
    training_kwargs["eval_strategy"] = (
        train_cfg["evaluation_strategy"]
    )

    evaluation_strategy_field = "eval_strategy"

elif "evaluation_strategy" in training_args_signature:
    training_kwargs["evaluation_strategy"] = (
        train_cfg["evaluation_strategy"]
    )

    evaluation_strategy_field = "evaluation_strategy"

else:
    raise RuntimeError(
        "Could not identify the TrainingArguments "
        "evaluation-strategy field."
    )

# ------------------------------------------------------------
# 4. Instantiate frozen TrainingArguments
# ------------------------------------------------------------
training_args = TrainingArguments(
    **training_kwargs
)

# ------------------------------------------------------------
# 5. Initialize Trainer
#
# NO training occurs in this cell.
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train_trainer,
    eval_dataset=hf_valid_trainer,
    data_collator=data_collator,
)

# ------------------------------------------------------------
# 6. Formal pre-training checks
# ------------------------------------------------------------
trainer_checks = {
    "trainer_initialized":
        trainer is not None,

    "train_n_13539":
        len(trainer.train_dataset) == 13_539,

    "validation_n_5859":
        len(trainer.eval_dataset) == 5_859,

    "max_steps_200":
        trainer.args.max_steps == 200,

    "learning_rate_1e_4":
        trainer.args.learning_rate == 1e-4,

    "train_batch_size_1":
        trainer.args.per_device_train_batch_size == 1,

    "eval_batch_size_1":
        trainer.args.per_device_eval_batch_size == 1,

    "gradient_accumulation_8":
        trainer.args.gradient_accumulation_steps == 8,

    "logging_steps_10":
        trainer.args.logging_steps == 10,

    "eval_steps_25":
        trainer.args.eval_steps == 25,

    "save_steps_50":
        trainer.args.save_steps == 50,

    "save_total_limit_4":
        trainer.args.save_total_limit == 4,

    "warmup_steps_10":
        trainer.args.warmup_steps == 10,

    "weight_decay_0_01":
        trainer.args.weight_decay == 0.01,

    "fp16_enabled":
        trainer.args.fp16 is True,

    "gradient_checkpointing_enabled":
        trainer.args.gradient_checkpointing is True,

    "seed_42":
        trainer.args.seed == 42,

    "data_seed_42":
        trainer.args.data_seed == 42,

    "load_best_model_disabled":
        trainer.args.load_best_model_at_end is False,

    "remove_unused_columns_false":
        trainer.args.remove_unused_columns is False,

    "candidate_checkpoints_frozen":
        ckpt_cfg["candidate_steps"]
        == [50, 100, 150, 200],

    "selection_metric_eval_loss":
        ckpt_cfg["selection_metric"]
        == "eval_loss",

    "selection_dataset_historical_validation":
        ckpt_cfg["selection_dataset"]
        == "historical_validation only",

    "future_metrics_forbidden":
        ckpt_cfg["future_window_metrics_allowed"]
        is False,
}

# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------
print("=" * 112)
print("DAY 15 — FINAL PRE-TRAINING CONFIGURATION GATE")
print("=" * 112)

for name, passed in trainer_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 112)

trainer_init_pass = all(
    trainer_checks.values()
)

print(
    "FINAL PRE-TRAINING STATUS:",
    "PASS" if trainer_init_pass else "FAIL"
)

print()

print(
    f"Train observations     : "
    f"{len(trainer.train_dataset):,}"
)

print(
    f"Validation observations: "
    f"{len(trainer.eval_dataset):,}"
)

print()

print(
    f"Training budget        : "
    f"{trainer.args.max_steps} steps"
)

print(
    f"Learning rate          : "
    f"{trainer.args.learning_rate}"
)

print(
    f"Per-device train batch : "
    f"{trainer.args.per_device_train_batch_size}"
)

print(
    f"Gradient accumulation  : "
    f"{trainer.args.gradient_accumulation_steps}"
)

print(
    f"Effective batch size   : "
    f"{trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps}"
)

print()

print(
    f"Evaluation strategy    : "
    f"{getattr(trainer.args, evaluation_strategy_field)}"
)

print(
    f"Evaluation interval    : "
    f"{trainer.args.eval_steps} steps"
)

print(
    f"Checkpoint interval    : "
    f"{trainer.args.save_steps} steps"
)

print(
    "Candidate checkpoints :",
    ckpt_cfg["candidate_steps"]
)

print()

print(
    "Checkpoint selection  : "
    "minimum historical-validation eval_loss"
)

print(
    f"Auto best-model loading: "
    f"{trainer.args.load_best_model_at_end}"
)

print(
    "Future metrics allowed:",
    ckpt_cfg["future_window_metrics_allowed"]
)

print()

print(
    f"Training output        : "
    f"{TRAINING_OUTPUT_DIR}"
)

print("=" * 112)

assert trainer_init_pass, (
    "DAY 15 STOP: final pre-training configuration "
    "does not match the frozen Day 14 protocol."
)

In [ ]:
# Execute frozen 200-step Historical Dataset B LoRA training

import time
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Record pre-training runtime state
# ------------------------------------------------------------
training_start_time = time.time()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

print("=" * 112)
print("DAY 15 — HISTORICAL DATASET B LoRA TRAINING")
print("=" * 112)

print(f"Model identity          : M_H")
print(f"Historical Train        : {len(hf_train_trainer):,}")
print(f"Historical Validation   : {len(hf_valid_trainer):,}")
print(f"Future observations used: 0")
print()

print(f"Maximum training steps  : {trainer.args.max_steps}")
print(f"Learning rate           : {trainer.args.learning_rate}")
print(f"Effective batch size    : "
      f"{trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps}")
print(f"Evaluation every        : {trainer.args.eval_steps} steps")
print(f"Checkpoint every        : {trainer.args.save_steps} steps")
print(f"Candidate checkpoints   : {ckpt_cfg['candidate_steps']}")
print()

print("Checkpoint selection    : historical-validation eval_loss only")
print("Future metrics allowed  : False")
print("-" * 112)

# ------------------------------------------------------------
# 2. Execute the EXACT frozen training budget
# ------------------------------------------------------------
train_result = trainer.train()

# ------------------------------------------------------------
# 3. Runtime summary
# ------------------------------------------------------------
training_end_time = time.time()
training_elapsed_seconds = (
    training_end_time - training_start_time
)

# ------------------------------------------------------------
# 4. Convert Trainer log history into a table
# ------------------------------------------------------------
training_log_df = pd.DataFrame(
    trainer.state.log_history
)

# ------------------------------------------------------------
# 5. Training-completion checks
# ------------------------------------------------------------
training_completion_checks = {
    "training_completed":
        train_result is not None,

    "global_step_200":
        trainer.state.global_step
        == train_cfg["max_steps"]
        == 200,

    "future_observations_used_zero":
        future_n_only == 20_002
        and len(hf_train_trainer) == 13_539
        and len(hf_valid_trainer) == 5_859,

    "trainer_max_steps_unchanged":
        trainer.args.max_steps == 200,

    "checkpoint_selection_rule_unchanged":
        ckpt_cfg["selection_metric"] == "eval_loss"
        and ckpt_cfg["selection_dataset"]
        == "historical_validation only",

    "future_metrics_still_forbidden":
        ckpt_cfg["future_window_metrics_allowed"]
        is False,

    "log_history_nonempty":
        len(training_log_df) > 0,
}

# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------
print()
print("=" * 112)
print("DAY 15 — 200-STEP TRAINING COMPLETION CHECKPOINT")
print("=" * 112)

for name, passed in training_completion_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 112)

training_complete_pass = all(
    training_completion_checks.values()
)

print(
    "FROZEN TRAINING STATUS:",
    "PASS" if training_complete_pass else "FAIL"
)

print()
print(
    f"Final global step       : "
    f"{trainer.state.global_step}"
)

print(
    f"Training runtime        : "
    f"{training_elapsed_seconds / 60:.2f} minutes"
)

if hasattr(train_result, "training_loss"):
    print(
        f"Reported training loss  : "
        f"{train_result.training_loss:.6f}"
    )

if torch.cuda.is_available():
    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    print(
        f"Peak GPU allocation     : "
        f"{peak_memory_gb:.2f} GB"
    )

print()
print(
    f"Trainer log entries     : "
    f"{len(training_log_df):,}"
)

print("=" * 112)

assert training_complete_pass, (
    "DAY 15 STOP: frozen 200-step training "
    "did not complete according to protocol."
)

In [ ]:
# Save training log and select checkpoint using frozen rule
import pandas as pd
from pathlib import Path

# 1. Save complete Trainer log history
training_log_df = pd.DataFrame(
    trainer.state.log_history
)

training_log_path = (
    DAY15_RESULT_DIR
    / "lora_training_log.csv"
)

training_log_df.to_csv(
    training_log_path,
    index=False,
)

# 2. Frozen candidate checkpoints
candidate_steps = ckpt_cfg["candidate_steps"]

candidate_rows = []

for step in candidate_steps:

    # Historical-validation eval_loss logged at this step
    matching_eval = [
        row
        for row in trainer.state.log_history
        if row.get("step") == step
        and "eval_loss" in row
    ]

    if len(matching_eval) != 1:
        eval_loss = None
    else:
        eval_loss = float(
            matching_eval[0]["eval_loss"]
        )

    checkpoint_path = (
        TRAINING_OUTPUT_DIR
        / f"checkpoint-{step}"
    )

    candidate_rows.append(
        {
            "checkpoint_step": step,
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_exists": checkpoint_path.exists(),
            "historical_validation_eval_loss": eval_loss,
        }
    )

checkpoint_selection_df = pd.DataFrame(
    candidate_rows
)

# 3. Formal candidate integrity checks

candidate_checks = {
    "four_candidate_checkpoints":
        checkpoint_selection_df.shape[0] == 4,

    "candidate_steps_exact":
        checkpoint_selection_df[
            "checkpoint_step"
        ].tolist()
        == [50, 100, 150, 200],

    "all_candidate_checkpoints_exist":
        checkpoint_selection_df[
            "checkpoint_exists"
        ].all(),

    "all_candidate_eval_losses_present":
        checkpoint_selection_df[
            "historical_validation_eval_loss"
        ].notna().all(),

    "selection_metric_eval_loss":
        ckpt_cfg["selection_metric"]
        == "eval_loss",

    "selection_direction_minimize":
        ckpt_cfg["selection_direction"]
        == "minimize",

    "selection_dataset_historical_validation_only":
        ckpt_cfg["selection_dataset"]
        == "historical_validation only",

    "future_metrics_not_used":
        ckpt_cfg[
            "future_window_metrics_allowed"
        ] is False,
}

# ------------------------------------------------------------
# 4. Select checkpoint using frozen rule
#
# Primary:
#   minimum historical-validation eval_loss
#
# Tie-break:
#   earliest checkpoint step
# ------------------------------------------------------------
eligible_candidates = (
    checkpoint_selection_df.loc[
        checkpoint_selection_df[
            "checkpoint_exists"
        ]
        &
        checkpoint_selection_df[
            "historical_validation_eval_loss"
        ].notna()
    ]
    .copy()
)

eligible_candidates = (
    eligible_candidates.sort_values(
        by=[
            "historical_validation_eval_loss",
            "checkpoint_step",
        ],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

selected_step = int(
    eligible_candidates.loc[
        0,
        "checkpoint_step"
    ]
)

selected_eval_loss = float(
    eligible_candidates.loc[
        0,
        "historical_validation_eval_loss"
    ]
)

selected_checkpoint_path = Path(
    eligible_candidates.loc[
        0,
        "checkpoint_path"
    ]
)

# Mark selection in formal table
checkpoint_selection_df[
    "selected"
] = (
    checkpoint_selection_df[
        "checkpoint_step"
    ] == selected_step
)

checkpoint_selection_df[
    "selection_rule"
] = (
    "minimum historical-validation eval_loss "
    "among frozen candidate checkpoints"
)

# 5. Save checkpoint-selection artifact

checkpoint_selection_path = (
    DAY15_RESULT_DIR
    / "checkpoint_selection.csv"
)

checkpoint_selection_df.to_csv(
    checkpoint_selection_path,
    index=False,
)

# 6. Selection checks

selection_checks = {
    **candidate_checks,

    "exactly_one_checkpoint_selected":
        int(
            checkpoint_selection_df[
                "selected"
            ].sum()
        ) == 1,

    "selected_checkpoint_exists":
        selected_checkpoint_path.exists(),

    "selected_checkpoint_is_candidate":
        selected_step in candidate_steps,

    "selected_loss_is_candidate_minimum":
        selected_eval_loss
        == checkpoint_selection_df[
            "historical_validation_eval_loss"
        ].min(),

    "training_log_saved":
        training_log_path.exists(),

    "checkpoint_selection_saved":
        checkpoint_selection_path.exists(),
}
print("=" * 112)
print("DAY 15 — FROZEN CHECKPOINT SELECTION")
print("=" * 112)

print("\nCandidate checkpoints:")
print(
    checkpoint_selection_df[
        [
            "checkpoint_step",
            "historical_validation_eval_loss",
            "checkpoint_exists",
            "selected",
        ]
    ].to_string(index=False)
)

print("\n" + "-" * 112)

for name, passed in selection_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 112)

checkpoint_selection_pass = all(
    selection_checks.values()
)

print(
    "CHECKPOINT SELECTION STATUS:",
    "PASS" if checkpoint_selection_pass else "FAIL"
)

print()
print(
    f"Selected checkpoint : "
    f"checkpoint-{selected_step}"
)

print(
    f"Selected eval_loss  : "
    f"{selected_eval_loss:.6f}"
)

print(
    f"Selected path       : "
    f"{selected_checkpoint_path}"
)

print()
print(
    "Selection rule      : "
    "minimum historical-validation eval_loss"
)

print(
    "Future metrics used : False"
)

print()
print(
    f"Saved training log  : "
    f"{training_log_path}"
)

print(
    f"Saved selection CSV : "
    f"{checkpoint_selection_path}"
)

print("=" * 112)

assert checkpoint_selection_pass, (
    "DAY 15 STOP: frozen checkpoint selection failed."
)

In [ ]:
# ============================================================
# DAY 15 — CELL 20
# Reload selected checkpoint and save frozen M_H adapter
# ============================================================

from pathlib import Path
import json
import torch

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ------------------------------------------------------------
# 1. Final frozen model directory
# ------------------------------------------------------------
FINAL_MODEL_DIR = (
    DAY15_MODEL_DIR
    / "frozen_selected_adapter"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Rebuild frozen quantized base model
# ------------------------------------------------------------
final_bnb_config = BitsAndBytesConfig(
    load_in_4bit=quant_cfg["load_in_4bit"],
    bnb_4bit_quant_type=quant_cfg["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=quant_cfg[
        "bnb_4bit_use_double_quant"
    ],
    bnb_4bit_compute_dtype=torch.float16,
)

selected_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_cfg["model_name"],
    quantization_config=final_bnb_config,
    device_map=base_model_cfg["device_map"],
    trust_remote_code=base_model_cfg["trust_remote_code"],
)

selected_base_model.config.use_cache = True

# ------------------------------------------------------------
# 3. Load selected Day 15 LoRA checkpoint
# ------------------------------------------------------------
selected_model = PeftModel.from_pretrained(
    selected_base_model,
    str(selected_checkpoint_path),
    is_trainable=False,
)

selected_model.eval()

# ------------------------------------------------------------
# 4. Save frozen adapter + tokenizer
# ------------------------------------------------------------
selected_model.save_pretrained(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

# ------------------------------------------------------------
# 5. Structural freeze checks
# ------------------------------------------------------------
trainable_after_freeze = sum(
    p.numel()
    for p in selected_model.parameters()
    if p.requires_grad
)

saved_files = sorted(
    [
        p.name
        for p in FINAL_MODEL_DIR.iterdir()
        if p.is_file()
    ]
)

freeze_checks = {
    "selected_checkpoint_step_200":
        selected_step == 200,

    "selected_checkpoint_exists":
        selected_checkpoint_path.exists(),

    "selected_model_loaded":
        selected_model is not None,

    "selected_model_eval_mode":
        selected_model.training is False,

    "no_trainable_parameters":
        trainable_after_freeze == 0,

    "final_model_directory_exists":
        FINAL_MODEL_DIR.exists(),

    "adapter_config_saved":
        (FINAL_MODEL_DIR / "adapter_config.json").exists(),

    "adapter_weights_saved":
        (
            (FINAL_MODEL_DIR / "adapter_model.safetensors").exists()
            or
            (FINAL_MODEL_DIR / "adapter_model.bin").exists()
        ),

    "tokenizer_saved":
        (FINAL_MODEL_DIR / "tokenizer_config.json").exists(),

    "future_metrics_unused":
        ckpt_cfg["future_window_metrics_allowed"] is False,
}

# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------
print("=" * 112)
print("DAY 15 — SELECTED MODEL RELOAD & FREEZE")
print("=" * 112)

for name, passed in freeze_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 112)

model_freeze_pass = all(
    freeze_checks.values()
)

print(
    "MODEL FREEZE STATUS:",
    "PASS" if model_freeze_pass else "FAIL"
)

print()
print(
    f"Selected checkpoint : "
    f"checkpoint-{selected_step}"
)

print(
    f"Selected eval_loss  : "
    f"{selected_eval_loss:.6f}"
)

print(
    f"Frozen model dir    : "
    f"{FINAL_MODEL_DIR}"
)

print(
    f"Trainable parameters: "
    f"{trainable_after_freeze:,}"
)

print()
print("Saved files:")
for name in saved_files:
    print(" -", name)

print("=" * 112)

assert model_freeze_pass, (
    "DAY 15 STOP: selected M_H could not be "
    "reloaded and frozen correctly."
)

In [ ]:
# Full Historical Validation inference using frozen M_H
import time
import pandas as pd
import torch
from tqdm.auto import tqdm

# 1. Frozen inference setup

selected_model.eval()

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

VALID_PREDICTIONS = set(
    infer_cfg["valid_predictions"]
)

# Conservative batch size for Tesla T4
INFERENCE_BATCH_SIZE = 4

# 2. Output paths

partial_prediction_path = (
    DAY15_RESULT_DIR
    / "historical_validation_predictions_partial.csv"
)

final_prediction_path = (
    DAY15_RESULT_DIR
    / "historical_validation_predictions.csv"
)

# 3. Prepare inference population
validation_inference_df = (
    historical_validation[
        [
            "observation_id",
            "channel_username",
            "message_id",
            "date",
            "text",
            "sentiment",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

validation_inference_df["date"] = (
    validation_inference_df["date"].astype(str)
)

if partial_prediction_path.exists():

    partial_df = pd.read_csv(
        partial_prediction_path
    )

    completed_n = len(partial_df)

    # Safety check: partial file must correspond to
    # the beginning of the exact frozen validation population.
    expected_ids = (
        validation_inference_df[
            "observation_id"
        ]
        .iloc[:completed_n]
        .astype(str)
        .tolist()
    )

    actual_ids = (
        partial_df[
            "observation_id"
        ]
        .astype(str)
        .tolist()
    )

    assert actual_ids == expected_ids, (
        "Partial prediction file does not match the "
        "frozen historical validation ordering."
    )

    prediction_records = (
        partial_df.to_dict("records")
    )

else:
    completed_n = 0
    prediction_records = []

inference_start_time = time.time()

for start_idx in tqdm(
    range(
        completed_n,
        len(validation_inference_df),
        INFERENCE_BATCH_SIZE,
    ),
    desc="Historical Validation Inference",
):

    end_idx = min(
        start_idx + INFERENCE_BATCH_SIZE,
        len(validation_inference_df),
    )

    batch_df = validation_inference_df.iloc[
        start_idx:end_idx
    ]

    prompts = [
        build_inference_prompt(text)
        for text in batch_df["text"]
    ]

    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )

    encoded = {
        k: v.to(selected_model.device)
        for k, v in encoded.items()
    }

    prompt_width = encoded[
        "input_ids"
    ].shape[1]

    with torch.no_grad():

        output_ids = selected_model.generate(
            **encoded,
            max_new_tokens=infer_cfg[
                "generation"
            ]["max_new_tokens"],
            do_sample=infer_cfg[
                "generation"
            ]["do_sample"],
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        :,
        prompt_width:
    ]

    decoded_predictions = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    decoded_predictions = [
        x.strip().lower()
        for x in decoded_predictions
    ]

    for local_i, (_, row) in enumerate(
        batch_df.iterrows()
    ):

        prediction = decoded_predictions[
            local_i
        ]

        prediction_records.append(
            {
                "observation_id":
                    row["observation_id"],

                "channel_username":
                    row["channel_username"],

                "message_id":
                    row["message_id"],

                "date":
                    row["date"],

                "true_label":
                    row["sentiment"],

                "predicted_label":
                    prediction,

                "prediction_valid":
                    prediction
                    in VALID_PREDICTIONS,
            }
        )
    # Periodic recovery save
    # Every ~200 observations

    if (
        len(prediction_records) % 200
        < INFERENCE_BATCH_SIZE
        or end_idx == len(validation_inference_df)
    ):
        pd.DataFrame(
            prediction_records
        ).to_csv(
            partial_prediction_path,
            index=False,
        )

# 6. Finalize inference results
validation_predictions = pd.DataFrame(
    prediction_records
)

validation_predictions.to_csv(
    final_prediction_path,
    index=False,
)

inference_elapsed_seconds = (
    time.time() - inference_start_time
)

# 7. Formal integrity checks

prediction_checks = {
    "prediction_n_5859":
        len(validation_predictions) == 5_859,

    "observation_ids_unique":
        validation_predictions[
            "observation_id"
        ].nunique() == 5_859,

    "observation_order_preserved":
        validation_predictions[
            "observation_id"
        ].astype(str).tolist()
        ==
        validation_inference_df[
            "observation_id"
        ].astype(str).tolist(),

    "all_predictions_nonmissing":
        validation_predictions[
            "predicted_label"
        ].notna().all(),

    "all_predictions_valid":
        validation_predictions[
            "prediction_valid"
        ].all(),

    "true_labels_valid":
        set(
            validation_predictions[
                "true_label"
            ].unique()
        )
        == {
            "negative",
            "neutral",
            "positive",
        },

    "future_observations_used_zero":
        (
            pd.to_datetime(
                validation_predictions["date"],
                utc=True,
            )
            <
            pd.Timestamp(
                "2026-01-01T00:00:00+00:00"
            )
        ).all(),

    "final_predictions_saved":
        final_prediction_path.exists(),
}

print("=" * 112)
print("DAY 15 — HISTORICAL VALIDATION INFERENCE CHECKPOINT")
print("=" * 112)

for name, passed in prediction_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 112)

historical_inference_pass = all(
    prediction_checks.values()
)

print(
    "HISTORICAL VALIDATION INFERENCE:",
    "PASS" if historical_inference_pass else "FAIL"
)

print()
print(
    f"Observations evaluated : "
    f"{len(validation_predictions):,}"
)

print(
    f"Valid predictions      : "
    f"{int(validation_predictions['prediction_valid'].sum()):,}"
)

print(
    f"Invalid predictions    : "
    f"{int((~validation_predictions['prediction_valid']).sum()):,}"
)

print(
    f"Inference runtime      : "
    f"{inference_elapsed_seconds / 60:.2f} minutes"
)

print()
print(
    f"Saved predictions      : "
    f"{final_prediction_path}"
)

print("=" * 112)

assert historical_inference_pass, (
    "DAY 15 STOP: historical-validation inference "
    "failed integrity checks."
)

In [ ]:
# ============================================================
# DAY 15 — CELL 22
# Compute Historical Validation baseline metrics
# ============================================================

import pandas as pd
import json

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. Load / use finalized predictions
# ------------------------------------------------------------
validation_predictions = pd.read_csv(
    final_prediction_path
)

LABEL_ORDER = [
    "negative",
    "neutral",
    "positive",
]

y_true = validation_predictions[
    "true_label"
].astype(str)

y_pred = validation_predictions[
    "predicted_label"
].astype(str)

# ------------------------------------------------------------
# 2. Core metrics
# ------------------------------------------------------------
accuracy = accuracy_score(
    y_true,
    y_pred,
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=LABEL_ORDER,
    average="macro",
    zero_division=0,
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=LABEL_ORDER,
    average="weighted",
    zero_division=0,
)

class_recalls = recall_score(
    y_true,
    y_pred,
    labels=LABEL_ORDER,
    average=None,
    zero_division=0,
)

# ------------------------------------------------------------
# 3. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABEL_ORDER,
)

confusion_df = pd.DataFrame(
    cm,
    index=[
        f"true_{label}"
        for label in LABEL_ORDER
    ],
    columns=[
        f"pred_{label}"
        for label in LABEL_ORDER
    ],
)

# ------------------------------------------------------------
# 4. Predicted-class distribution
# ------------------------------------------------------------
pred_counts = (
    y_pred
    .value_counts()
    .reindex(
        LABEL_ORDER,
        fill_value=0,
    )
)

pred_distribution_df = pd.DataFrame(
    {
        "predicted_label":
            LABEL_ORDER,

        "count":
            [
                int(
                    pred_counts[label]
                )
                for label in LABEL_ORDER
            ],
    }
)

pred_distribution_df[
    "proportion"
] = (
    pred_distribution_df["count"]
    / len(validation_predictions)
)

# ------------------------------------------------------------
# 5. Metrics table
# ------------------------------------------------------------
metrics_records = [
    {
        "metric": "accuracy",
        "value": float(accuracy),
    },
    {
        "metric": "macro_f1",
        "value": float(macro_f1),
    },
    {
        "metric": "weighted_f1",
        "value": float(weighted_f1),
    },
    {
        "metric": "negative_recall",
        "value": float(class_recalls[0]),
    },
    {
        "metric": "neutral_recall",
        "value": float(class_recalls[1]),
    },
    {
        "metric": "positive_recall",
        "value": float(class_recalls[2]),
    },
]

historical_validation_metrics_df = (
    pd.DataFrame(
        metrics_records
    )
)

# ------------------------------------------------------------
# 6. Save formal artifacts
# ------------------------------------------------------------
metrics_path = (
    DAY15_RESULT_DIR
    / "historical_validation_metrics.csv"
)

confusion_path = (
    DAY15_RESULT_DIR
    / "historical_validation_confusion.csv"
)

pred_distribution_path = (
    DAY15_RESULT_DIR
    / "historical_validation_pred_distribution.csv"
)

historical_validation_metrics_df.to_csv(
    metrics_path,
    index=False,
)

confusion_df.to_csv(
    confusion_path,
)

pred_distribution_df.to_csv(
    pred_distribution_path,
    index=False,
)

# ------------------------------------------------------------
# 7. Formal checks
# ------------------------------------------------------------
metric_checks = {
    "prediction_n_5859":
        len(validation_predictions)
        == 5_859,

    "all_three_true_labels_present":
        set(y_true.unique())
        == set(LABEL_ORDER),

    "all_three_pred_labels_valid":
        set(y_pred.unique()).issubset(
            set(LABEL_ORDER)
        ),

    "accuracy_valid":
        0.0 <= accuracy <= 1.0,

    "macro_f1_valid":
        0.0 <= macro_f1 <= 1.0,

    "weighted_f1_valid":
        0.0 <= weighted_f1 <= 1.0,

    "all_recalls_valid":
        all(
            0.0 <= x <= 1.0
            for x in class_recalls
        ),

    "confusion_matrix_total_5859":
        int(cm.sum()) == 5_859,

    "prediction_distribution_total_5859":
        int(
            pred_distribution_df[
                "count"
            ].sum()
        )
        == 5_859,

    "metrics_saved":
        metrics_path.exists(),

    "confusion_saved":
        confusion_path.exists(),

    "prediction_distribution_saved":
        pred_distribution_path.exists(),
}

# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------
print("=" * 112)
print("DAY 15 — HISTORICAL VALIDATION BASELINE METRICS")
print("=" * 112)

for name, passed in metric_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 112)

historical_metrics_pass = all(
    metric_checks.values()
)

print(
    "HISTORICAL VALIDATION METRICS:",
    "PASS" if historical_metrics_pass else "FAIL"
)

print()

print(f"Accuracy        : {accuracy:.6f}")
print(f"Macro-F1        : {macro_f1:.6f}")
print(f"Weighted-F1     : {weighted_f1:.6f}")

print()

print(
    f"Negative recall : "
    f"{class_recalls[0]:.6f}"
)

print(
    f"Neutral recall  : "
    f"{class_recalls[1]:.6f}"
)

print(
    f"Positive recall : "
    f"{class_recalls[2]:.6f}"
)

print()
print("Confusion matrix:")
print(confusion_df)

print()
print("Predicted class distribution:")
print(
    pred_distribution_df.to_string(
        index=False
    )
)

print()
print("Saved:")
print(" -", metrics_path)
print(" -", confusion_path)
print(" -", pred_distribution_path)

print("=" * 112)

assert historical_metrics_pass, (
    "DAY 15 STOP: historical-validation "
    "metric computation failed."
)

In [ ]:
# ============================================================
# DAY 15 — CELL 23
# Save final frozen M_H model manifest
# ============================================================

import json
from datetime import datetime, timezone
from pathlib import Path

# ------------------------------------------------------------
# 1. Clean up completed partial-inference artifact
# ------------------------------------------------------------
if partial_prediction_path.exists():
    partial_prediction_path.unlink()

partial_prediction_removed = (
    not partial_prediction_path.exists()
)

# ------------------------------------------------------------
# 2. Runtime metadata
# ------------------------------------------------------------
runtime_metadata = {
    "python_version":
        sys.version.split()[0],

    "pytorch_version":
        torch.__version__,

    "transformers_version":
        transformers.__version__,

    "datasets_version":
        datasets.__version__,

    "peft_version":
        peft.__version__,

    "accelerate_version":
        accelerate.__version__,

    "cuda_available":
        bool(torch.cuda.is_available()),

    "cuda_runtime":
        torch.version.cuda,

    "gpu_name":
        (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),

    "gpu_total_memory_gb":
        (
            float(
                torch.cuda.get_device_properties(0).total_memory
                / (1024 ** 3)
            )
            if torch.cuda.is_available()
            else None
        ),
}

# ------------------------------------------------------------
# 3. Historical baseline metrics
# ------------------------------------------------------------
historical_baseline_metrics = {
    "n_observations": 5_859,

    "accuracy":
        float(accuracy),

    "macro_f1":
        float(macro_f1),

    "weighted_f1":
        float(weighted_f1),

    "negative_recall":
        float(class_recalls[0]),

    "neutral_recall":
        float(class_recalls[1]),

    "positive_recall":
        float(class_recalls[2]),

    "predicted_class_distribution": {
        row["predicted_label"]: {
            "count": int(row["count"]),
            "proportion": float(row["proportion"]),
        }
        for _, row in pred_distribution_df.iterrows()
    },
}

# ------------------------------------------------------------
# 4. Final frozen-model manifest
# ------------------------------------------------------------
frozen_model_manifest = {
    "day": 15,
    "artifact": "frozen_model_manifest",

    "project":
        "Financial Language Models under Temporal Distribution Shift",

    "model_identity": {
        "model_id": "M_H",
        "role":
            "Historical Dataset B model for primary within-dataset temporal benchmark",

        "base_model":
            base_model_cfg["model_name"],

        "adapter_initialization":
            "fresh LoRA initialized from common base model",

        "day04_adapter_loaded":
            False,

        "frozen_model_directory":
            str(FINAL_MODEL_DIR),

        "freeze_status":
            "FROZEN_BEFORE_FUTURE_EVALUATION",
    },

    "training_population": {
        "split":
            "historical_train",

        "n_observations":
            int(len(historical_train)),

        "start_timestamp_utc":
            historical_train["date"].min().isoformat(),

        "end_timestamp_utc":
            historical_train["date"].max().isoformat(),

        "future_observations_used":
            0,
    },

    "validation_population": {
        "split":
            "historical_validation",

        "n_observations":
            int(len(historical_validation)),

        "start_timestamp_utc":
            historical_validation["date"].min().isoformat(),

        "end_timestamp_utc":
            historical_validation["date"].max().isoformat(),

        "future_observations_used":
            0,
    },

    "quantization": quant_cfg,

    "lora_configuration": {
        "r":
            lora_cfg["r"],

        "lora_alpha":
            lora_cfg["lora_alpha"],

        "lora_dropout":
            lora_cfg["lora_dropout"],

        "target_modules":
            lora_cfg["target_modules"],

        "bias":
            lora_cfg["bias"],

        "task_type":
            lora_cfg["task_type"],
    },

    "sequence_protocol": {
        "max_seq_length":
            seq_cfg["max_seq_length"],

        "prompt_template":
            seq_cfg["prompt_template"],

        "target_labels":
            seq_cfg["target_labels"],

        "loss_masking":
            seq_cfg["loss_masking"],

        "training_pad_token":
            seq_cfg["training_pad_token"],

        "padding_label_value":
            seq_cfg["padding_label_value"],
    },

    "training_protocol": {
        "max_steps":
            train_cfg["max_steps"],

        "learning_rate":
            train_cfg["learning_rate"],

        "per_device_train_batch_size":
            train_cfg["per_device_train_batch_size"],

        "gradient_accumulation_steps":
            train_cfg["gradient_accumulation_steps"],

        "effective_train_batch_size":
            train_cfg["effective_train_batch_size"],

        "eval_steps":
            train_cfg["eval_steps"],

        "save_steps":
            train_cfg["save_steps"],

        "warmup_steps":
            train_cfg["warmup_steps"],

        "weight_decay":
            train_cfg["weight_decay"],

        "seed":
            train_cfg["seed"],

        "data_seed":
            train_cfg["data_seed"],

        "training_runtime_minutes":
            float(training_elapsed_seconds / 60),

        "peak_gpu_allocation_gb":
            float(peak_memory_gb),
    },

    "checkpoint_selection": {
        "candidate_steps":
            candidate_steps,

        "selection_dataset":
            ckpt_cfg["selection_dataset"],

        "selection_metric":
            ckpt_cfg["selection_metric"],

        "selection_direction":
            ckpt_cfg["selection_direction"],

        "selected_checkpoint_step":
            int(selected_step),

        "selected_checkpoint_path":
            str(selected_checkpoint_path),

        "selected_historical_validation_eval_loss":
            float(selected_eval_loss),

        "future_metrics_used":
            False,
    },

    "historical_validation_baseline":
        historical_baseline_metrics,

    "inference_protocol": {
        "model_mode":
            infer_cfg["model_mode"],

        "padding_side":
            infer_cfg["tokenizer_padding_side"],

        "do_sample":
            infer_cfg["generation"]["do_sample"],

        "max_new_tokens":
            infer_cfg["generation"]["max_new_tokens"],

        "valid_predictions":
            infer_cfg["valid_predictions"],
    },

    "future_firewall": {
        "future_windows":
            ["F1", "F2", "F3", "F4", "F5", "F6"],

        "future_observations_available":
            20_002,

        "future_observations_used_for_training":
            0,

        "future_observations_used_for_validation":
            0,

        "future_metrics_used_for_checkpoint_selection":
            False,

        "future_evaluation_status":
            "NOT_STARTED_DAY15",
    },

    "runtime":
        runtime_metadata,

    "provenance": {
        "historical_train_manifest":
            str(train_manifest_path),

        "historical_validation_manifest":
            str(validation_manifest_path),

        "training_data_audit":
            str(training_audit_path),

        "lora_training_log":
            str(training_log_path),

        "checkpoint_selection":
            str(checkpoint_selection_path),

        "historical_validation_predictions":
            str(final_prediction_path),

        "historical_validation_metrics":
            str(metrics_path),

        "historical_validation_confusion":
            str(confusion_path),

        "historical_validation_pred_distribution":
            str(pred_distribution_path),
    },

    "created_utc":
        datetime.now(timezone.utc).isoformat(),
}

# ------------------------------------------------------------
# 5. Save manifest
# ------------------------------------------------------------
frozen_manifest_path = (
    DAY15_RESULT_DIR
    / "frozen_model_manifest.json"
)

with open(
    frozen_manifest_path,
    "w",
) as f:

    json.dump(
        frozen_model_manifest,
        f,
        indent=2,
    )

# ------------------------------------------------------------
# 6. Formal checks
# ------------------------------------------------------------
frozen_manifest_checks = {
    "manifest_saved":
        frozen_manifest_path.exists(),

    "model_identity_M_H":
        frozen_model_manifest[
            "model_identity"
        ]["model_id"] == "M_H",

    "fresh_lora_confirmed":
        frozen_model_manifest[
            "model_identity"
        ]["day04_adapter_loaded"] is False,

    "selected_checkpoint_200":
        frozen_model_manifest[
            "checkpoint_selection"
        ]["selected_checkpoint_step"] == 200,

    "historical_train_n_13539":
        frozen_model_manifest[
            "training_population"
        ]["n_observations"] == 13_539,

    "historical_validation_n_5859":
        frozen_model_manifest[
            "validation_population"
        ]["n_observations"] == 5_859,

    "future_training_use_zero":
        frozen_model_manifest[
            "future_firewall"
        ]["future_observations_used_for_training"] == 0,

    "future_validation_use_zero":
        frozen_model_manifest[
            "future_firewall"
        ]["future_observations_used_for_validation"] == 0,

    "future_selection_use_false":
        frozen_model_manifest[
            "future_firewall"
        ]["future_metrics_used_for_checkpoint_selection"] is False,

    "future_evaluation_not_started":
        frozen_model_manifest[
            "future_firewall"
        ]["future_evaluation_status"]
        == "NOT_STARTED_DAY15",

    "model_frozen_before_future":
        frozen_model_manifest[
            "model_identity"
        ]["freeze_status"]
        == "FROZEN_BEFORE_FUTURE_EVALUATION",

    "historical_macro_f1_recorded":
        abs(
            frozen_model_manifest[
                "historical_validation_baseline"
            ]["macro_f1"]
            - macro_f1
        ) < 1e-12,

    "partial_prediction_file_removed":
        partial_prediction_removed,
}

# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------
print("=" * 112)
print("DAY 15 — FROZEN MODEL MANIFEST CHECKPOINT")
print("=" * 112)

for name, passed in frozen_manifest_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("-" * 112)

frozen_manifest_pass = all(
    frozen_manifest_checks.values()
)

print(
    "FROZEN MODEL MANIFEST:",
    "PASS" if frozen_manifest_pass else "FAIL"
)

print()

print("Model identity       : M_H")
print(
    f"Selected checkpoint  : checkpoint-{selected_step}"
)
print(
    f"Selected eval_loss   : {selected_eval_loss:.6f}"
)
print(
    f"Historical Macro-F1  : {macro_f1:.6f}"
)
print(
    f"Historical Accuracy  : {accuracy:.6f}"
)
print(
    "Future evaluation    : NOT STARTED"
)
print(
    "Model status         : FROZEN BEFORE FUTURE EVALUATION"
)

print()
print(
    "Saved manifest       :",
    frozen_manifest_path
)

print(
    "Partial inference CSV:",
    "REMOVED" if partial_prediction_removed else "STILL PRESENT"
)

print("=" * 112)

assert frozen_manifest_pass, (
    "DAY 15 STOP: frozen model manifest failed."
)

In [ ]:
# ============================================================
# DAY 15 — CELL 24
# Final artifact checkpoint and terminal status
# ============================================================

import json
from datetime import datetime, timezone
from pathlib import Path

# ------------------------------------------------------------
# 1. Create protocol-required historical metrics JSON
# ------------------------------------------------------------
historical_metrics_json = {
    "day": 15,
    "artifact": "historical_validation_metrics",
    "model_id": "M_H",
    "split": "historical_validation",
    "n_observations": 5_859,

    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),

    "class_recall": {
        "negative": float(class_recalls[0]),
        "neutral": float(class_recalls[1]),
        "positive": float(class_recalls[2]),
    },

    "predicted_class_distribution": {
        row["predicted_label"]: {
            "count": int(row["count"]),
            "proportion": float(row["proportion"]),
        }
        for _, row in pred_distribution_df.iterrows()
    },

    "future_observations_used": 0,
    "selected_checkpoint_step": int(selected_step),
    "selected_checkpoint_eval_loss": float(selected_eval_loss),

    "created_utc":
        datetime.now(timezone.utc).isoformat(),
}

historical_metrics_json_path = (
    DAY15_RESULT_DIR
    / "historical_validation_metrics.json"
)

with open(
    historical_metrics_json_path,
    "w",
) as f:
    json.dump(
        historical_metrics_json,
        f,
        indent=2,
    )

# ------------------------------------------------------------
# 2. Create protocol-required confusion-matrix CSV
# ------------------------------------------------------------
formal_confusion_df = (
    confusion_df
    .copy()
    .reset_index()
    .rename(
        columns={
            "index": "true_label"
        }
    )
)

historical_confusion_matrix_path = (
    DAY15_RESULT_DIR
    / "historical_validation_confusion_matrix.csv"
)

formal_confusion_df.to_csv(
    historical_confusion_matrix_path,
    index=False,
)

# ------------------------------------------------------------
# 3. Create Day 15 summary
# ------------------------------------------------------------
day15_summary = {
    "day": 15,
    "project":
        "Financial Language Models under Temporal Distribution Shift",

    "status":
        "PASS — HISTORICAL LoRA TRAINED AND FROZEN",

    "model": {
        "model_id": "M_H",
        "base_model": base_model_cfg["model_name"],
        "training_population_n": 13_539,
        "validation_population_n": 5_859,
        "future_population_n": 20_002,
        "future_observations_used": 0,
    },

    "training": {
        "max_steps": 200,
        "final_global_step": 200,
        "selected_checkpoint_step": int(selected_step),
        "selected_checkpoint_eval_loss":
            float(selected_eval_loss),
        "training_runtime_minutes":
            float(training_elapsed_seconds / 60),
    },

    "historical_validation": {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "negative_recall": float(class_recalls[0]),
        "neutral_recall": float(class_recalls[1]),
        "positive_recall": float(class_recalls[2]),
    },

    "model_freeze": {
        "frozen_model_directory":
            str(FINAL_MODEL_DIR),

        "trainable_parameters_after_freeze":
            int(trainable_after_freeze),

        "freeze_status":
            "FROZEN_BEFORE_FUTURE_EVALUATION",
    },

    "future_firewall": {
        "future_evaluation_started": False,
        "future_metrics_used_for_training": False,
        "future_metrics_used_for_validation": False,
        "future_metrics_used_for_checkpoint_selection": False,
        "next_stage":
            "Day 16 — frozen M_H evaluation on F1–F6",
    },

    "created_utc":
        datetime.now(timezone.utc).isoformat(),
}

day15_summary_path = (
    DAY15_RESULT_DIR
    / "day15_summary.json"
)

with open(
    day15_summary_path,
    "w",
) as f:
    json.dump(
        day15_summary,
        f,
        indent=2,
    )

# ------------------------------------------------------------
# 4. Required artifact checklist
# ------------------------------------------------------------
required_artifacts = {
    "historical_train_manifest.json":
        DAY15_RESULT_DIR / "historical_train_manifest.json",

    "historical_validation_manifest.json":
        DAY15_RESULT_DIR / "historical_validation_manifest.json",

    "training_data_audit.json":
        DAY15_RESULT_DIR / "training_data_audit.json",

    "lora_training_log.csv":
        DAY15_RESULT_DIR / "lora_training_log.csv",

    "checkpoint_selection.csv":
        DAY15_RESULT_DIR / "checkpoint_selection.csv",

    "historical_validation_predictions.csv":
        DAY15_RESULT_DIR / "historical_validation_predictions.csv",

    "historical_validation_metrics.json":
        historical_metrics_json_path,

    "historical_validation_confusion_matrix.csv":
        historical_confusion_matrix_path,

    "frozen_model_manifest.json":
        frozen_manifest_path,

    "day15_summary.json":
        day15_summary_path,
}

artifact_checks = {
    name: path.exists()
    for name, path in required_artifacts.items()
}

# ------------------------------------------------------------
# 5. Frozen-model directory checks
# ------------------------------------------------------------
model_checks = {
    "frozen_model_directory_exists":
        FINAL_MODEL_DIR.exists(),

    "adapter_config_exists":
        (FINAL_MODEL_DIR / "adapter_config.json").exists(),

    "adapter_weights_exist":
        (
            (FINAL_MODEL_DIR / "adapter_model.safetensors").exists()
            or
            (FINAL_MODEL_DIR / "adapter_model.bin").exists()
        ),

    "tokenizer_config_exists":
        (FINAL_MODEL_DIR / "tokenizer_config.json").exists(),

    "trainable_parameters_zero":
        trainable_after_freeze == 0,

    "selected_checkpoint_200":
        selected_step == 200,

    "selected_checkpoint_exists":
        selected_checkpoint_path.exists(),
}

# ------------------------------------------------------------
# 6. Scientific acceptance criteria
# ------------------------------------------------------------
acceptance_checks = {
    "historical_train_n_13539":
        len(historical_train) == 13_539,

    "historical_validation_n_5859":
        len(historical_validation) == 5_859,

    "future_n_20002":
        20_002 == 20_002,

    "historical_validation_predictions_n_5859":
        len(validation_predictions) == 5_859,

    "all_predictions_valid":
        validation_predictions[
            "prediction_valid"
        ].all(),

    "training_completed_200_steps":
        trainer.state.global_step == 200,

    "checkpoint_selected_by_historical_eval_loss":
        selected_eval_loss
        == checkpoint_selection_df[
            "historical_validation_eval_loss"
        ].min(),

    "future_used_for_training_zero":
        frozen_model_manifest[
            "future_firewall"
        ]["future_observations_used_for_training"] == 0,

    "future_used_for_validation_zero":
        frozen_model_manifest[
            "future_firewall"
        ]["future_observations_used_for_validation"] == 0,

    "future_used_for_selection_false":
        frozen_model_manifest[
            "future_firewall"
        ]["future_metrics_used_for_checkpoint_selection"] is False,

    "future_evaluation_not_started":
        frozen_model_manifest[
            "future_firewall"
        ]["future_evaluation_status"]
        == "NOT_STARTED_DAY15",

    "model_frozen_before_future":
        frozen_model_manifest[
            "model_identity"
        ]["freeze_status"]
        == "FROZEN_BEFORE_FUTURE_EVALUATION",
}

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------
all_artifacts_pass = all(
    artifact_checks.values()
)

all_model_checks_pass = all(
    model_checks.values()
)

all_acceptance_pass = all(
    acceptance_checks.values()
)

day15_final_pass = (
    all_artifacts_pass
    and all_model_checks_pass
    and all_acceptance_pass
)

# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------
print("=" * 116)
print("DAY 15 — FINAL ARTIFACT CHECKPOINT")
print("=" * 116)

print("\nRequired artifacts:")
for name, passed in artifact_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("\nFrozen model:")
for name, passed in model_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("\nAcceptance criteria:")
for name, passed in acceptance_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL':<7} — {name}"
    )

print("\n" + "=" * 116)

print(
    "DAY 15 STATUS:",
    (
        "PASS — HISTORICAL LoRA TRAINED AND FROZEN"
        if day15_final_pass
        else "FAIL"
    )
)

print("-" * 116)

print(f"Historical Train       : {len(historical_train):,}")
print(f"Historical Validation  : {len(historical_validation):,}")
print(f"Selected checkpoint    : checkpoint-{selected_step}")
print(f"Selected eval_loss     : {selected_eval_loss:.6f}")

print()
print(f"Accuracy               : {accuracy:.6f}")
print(f"Macro-F1               : {macro_f1:.6f}")
print(f"Weighted-F1            : {weighted_f1:.6f}")
print(f"Negative recall        : {class_recalls[0]:.6f}")
print(f"Neutral recall         : {class_recalls[1]:.6f}")
print(f"Positive recall        : {class_recalls[2]:.6f}")

print()
print("Future observations used: 0")
print("Future evaluation       : NOT STARTED")
print("Model status             : FROZEN")

print()
print("Next stage:")
print(
    "Day 16 — Evaluate frozen M_H on strictly future F1–F6."
)

print("=" * 116)

assert day15_final_pass, (
    "DAY 15 FINAL CHECKPOINT FAILED."
)